# Kinyarwanda News Topic Classification

**Task:** assign a Kinyarwanda news article to one of 14 topics (politics, sport, economy,
health, entertainment, history, technology, tourism, culture, fashion, religion,
environment, education, relationship).

**Dataset:** KINNEWS (Niyongabo et al., 2020), raw version: 17,014 training and 4,254 test
articles, MIT licence.

**Models:** TF-IDF + logistic regression on word and on character n-grams, a BiLSTM with
random or fastText embeddings, and fine-tuned XLM-R and AfroXLMR, against a majority-class
floor.

| RQ | Question | Section |
|---|---|---|
| **RQ1** | How far can a sparse linear model go, and do character n-grams beat word n-grams? | 3 |
| **RQ2** | Do in-domain fastText embeddings help a BiLSTM over random initialisation? | 3 |
| **RQ3** | Does African-centric pretraining (AfroXLMR) beat general multilingual pretraining (XLM-R)? | 4 |
| **RQ4** | What does the best model still get wrong, and why? | 6–7 |

| Section | Contents |
|---|---|
| 0 | Setup |
| 1 | Dataset: loading, audit, splits, exploratory analysis, preprocessing |
| 2 | Baselines (B0, B1) |
| 3 | Classical and recurrent experiments (E1, E2a, E2b) |
| 4 | Transformer experiments (E3, E4) |
| 5 | Ablations (A1, A2) and seed variance |
| 6 | Evaluation of the selected model |
| 7 | Error analysis |
| 8 | Export for the web app |
| 9 | Results summary |

**Running it:** on Kaggle (GPU T4, Internet on) or Colab (T4 GPU), set
`CONFIG["fast_mode"] = False` in 0.2 and run all cells (about 1 h 45 min on one T4).
`fast_mode = True` is a ~10-minute CPU smoke test; its results go to `results/runs/_fast/`
and are never reported.

**How results are recorded:** every experiment writes `results/runs/<id>.json`, and the
tables and "What we learned" findings are generated from those files. Discussion,
limitations and references are in the report.

| | |
|---|---|
| **Report** | `report/Kinyarwanda_News_Classification_Report.docx` |
| **Live app** | https://huggingface.co/spaces/Elvin100S/kinyarwanda-news-classifier |
| **GitHub** | https://github.com/Elvin100s/kinyarwanda-news-classification |
| **Video walkthrough** | `[[TODO: video link]]` |

## 0. Setup

### 0.1 Environment and dependencies

Detects Colab, Kaggle or a local machine. On Colab and Kaggle it installs the versions
pinned in `requirements.txt` (except `torch`, which the platform matches to its GPU
driver), so results do not drift when the platform upgrades its libraries.

In [ ]:
import os
import sys
import platform
import subprocess

# Kaggle images also contain /content, so check Kaggle first.
IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = not IN_KAGGLE and ("google.colab" in sys.modules or os.path.exists("/content"))
HOSTED = IN_COLAB or IN_KAGGLE

# One GPU: "T4 x2" would run DataParallel and double the effective batch.
if IN_KAGGLE:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

PINNED = [
    "torch==2.14.0",
    "transformers==4.57.6",
    "datasets==4.8.5",
    "accelerate==1.15.0",
    "gensim==4.4.0",
    "scikit-learn==1.8.0",
    "jupytext==1.19.5",
]

if HOSTED:
    # torch is left as installed: the platform matches it to the GPU driver.
    to_install = [p for p in PINNED if not p.startswith("torch==")]
    where = "Google Colab" if IN_COLAB else "Kaggle"
    print(f"Running on {where} -> installing {len(to_install)} pinned packages (~2 min).")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *to_install], check=True)
    print("If the platform warns about a restart, restart the session and run again.")
else:
    print("Running locally -> assuming the pinned virtualenv is already active.")

print(f"python   : {platform.python_version()}")
print(f"platform : {platform.platform()}")
print(f"in_colab : {IN_COLAB}")
print(f"in_kaggle: {IN_KAGGLE}")

### 0.2 Configuration

Every tunable value (data, preprocessing, model hyperparameters, evaluation) lives in one
`CONFIG` dict, and each result file records the hyperparameters it used, so any number
can be traced to the settings that produced it. `fast_mode` subsamples the data and shortens training for a
quick smoke test; the assertions at the end stop smoke-test settings leaking into a full
run.

In [ ]:
CONFIG = {
    # ---------- identity ----------
    "project": "kinyarwanda-news-classification",
    "seed": 42,

    "storage": "drive",
    "stream_corpus": False,
    # True = ~10-minute CPU smoke test (never reported); False = full run on a GPU.
    "fast_mode": True,

    # ---------- data ----------
    "data": {
        "hf_id": "kinnews_kirnews",
        "hf_config": "kinnews_raw",
        "num_labels": 14,
        "val_size": 0.10,          # stratified slice of the official train split
        "label_column": "label",
        "title_column": "title",
        "content_column": "content",
    },

    # ---------- quality audit ----------
    "audit": {
        "near_dup_threshold": 0.90,   # TF-IDF cosine above this = near-duplicate
        "min_words": 10,              # articles shorter than this are flagged
        "label_spot_check_n": 20,     # articles shown for manual review by the student
    },

    # ---------- preprocessing ----------
    "preprocess": {
        # "title_content" = title + " . " + content (default), "title_only" = ablation A1
        "text_mode": "title_content",
        "url_token": "<url>",
        "email_token": "<email>",
        "num_token": "<num>",
    },

    # ---------- B1 / E1: sparse linear models ----------
    "classical": {
        "word_ngram_range": (1, 2),
        "word_min_df": 2,
        "word_max_features": 200_000,
        "char_ngram_range": (2, 5),
        "char_analyzer": "char_wb",
        "char_min_df": 3,
        "char_max_features": 300_000,
        "sublinear_tf": True,
        "C_grid": [0.1, 1.0, 10.0],   # tuned on validation only
        "class_weight": "balanced",
        "max_iter": 2000,
        "top_features_per_class": 15,
    },

    # ---------- E2: BiLSTM ----------
    "bilstm": {
        "embed_dim": 300,
        "hidden_dim": 128,            # per direction; concatenated -> 256
        "num_layers": 1,
        "bidirectional": True,
        "dropout": 0.3,
        "max_tokens": 400,
        "min_freq": 2,                # vocabulary built on the train split only
        "lr": 1e-3,
        "batch_size": 64,
        "epochs": 15,
        "patience": 3,                # early stopping on validation macro-F1
        "grad_clip": 1.0,
        "class_weighted_loss": True,
    },

    # ---------- E2b: fastText embeddings ----------
    "fasttext": {
        "sg": 1,                      # skip-gram (Bojanowski et al., 2017)
        "vector_size": 300,
        "window": 5,
        "min_count": 2,
        "epochs": 10,
        "negative": 5,
        "min_n": 3,                   # character n-gram range for subword vectors
        "max_n": 6,
        # gensim's default (2M buckets, ~2.4 GB) is far more than this corpus needs.
        "bucket": 200_000,
        # One thread: multi-threaded training is not reproducible.
        "workers": 1,
        "neighbour_probe_words": [],  # filled in Phase 4 with real Kinyarwanda words
    },

    # ---------- E3 / E4: Transformers ----------
    "transformer": {
        "models": {
            "E3": "xlm-roberta-base",
            "E4": "Davlan/afro-xlmr-base",
        },
        "lr": 2e-5,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "epochs": 3,
        "max_length": 512,
        "per_device_batch_size": 16,
        "class_weighted_loss": True,
        "grad_accum_steps": 1,        # effective batch 16
        "fp16": True,                 # GPU only; disabled automatically on CPU
        "metric_for_best_model": "eval_macro_f1",
    },

    # ---------- seed variance (Section 5.2) ----------
    "seed_variance": {
        "extra_seeds": [43, 44],
        "experiments": ["E2a", "E2b", "E3", "E4"],
    },

    # ---------- ablations ----------
    "ablations": {
        "A1_text_mode": "title_only",
        "A2_max_length": 256,
    },

    # ---------- evaluation ----------
    "eval": {
        "primary_metric": "macro_f1",
        "bootstrap_n": 1000,
        "bootstrap_ci": 0.95,
        "n_confident_mistakes": 10,
        "n_taxonomy_errors": 30,
        "length_buckets": [0, 100, 250, 500, 10**9],
    },
}

# The FAST_MODE environment override must run before the overrides below.
if "FAST_MODE" in os.environ:
    CONFIG["fast_mode"] = os.environ["FAST_MODE"].lower() not in ("0", "false", "no")

if CONFIG["fast_mode"]:
    CONFIG["fast"] = {
        "n_train": 800,
        "n_val": 200,
        "n_test": 200,
    }
    CONFIG["classical"]["C_grid"] = [1.0]
    CONFIG["bilstm"]["epochs"] = 2
    CONFIG["bilstm"]["max_tokens"] = 128
    CONFIG["fasttext"]["epochs"] = 2
    CONFIG["transformer"]["epochs"] = 1
    # The tiny stand-in encoder only supports short sequences.
    CONFIG["transformer"]["max_length"] = 32
    CONFIG["eval"]["bootstrap_n"] = 100
    CONFIG["ablations"]["A2_max_length"] = 16
    CONFIG["seed_variance"]["extra_seeds"] = [43]
    CONFIG["transformer"]["models"] = {
        "E3": "hf-internal-testing/tiny-xlm-roberta",
        "E4": "hf-internal-testing/tiny-xlm-roberta",
    }

# A full run must not carry smoke-test settings.
if not CONFIG["fast_mode"]:
    assert CONFIG["eval"]["bootstrap_n"] == 1000, "fast-mode bootstrap leaked into a full run"
    assert len(CONFIG["classical"]["C_grid"]) == 3, "fast-mode C grid leaked into a full run"
    assert CONFIG["transformer"]["max_length"] == 512, "fast-mode max_length leaked into a full run"
    assert "fast" not in CONFIG, "fast-mode subsample sizes leaked into a full run"

print("fast_mode:", CONFIG["fast_mode"])
if CONFIG["fast_mode"]:
    print("  !! Numbers produced in this mode are a smoke test only and are never reported.")
print(f"seed: {CONFIG['seed']}   primary metric: {CONFIG['eval']['primary_metric']}")

### 0.3 Paths and storage

Creates the output folders. On Colab, results and checkpoints go to Google Drive so a
dropped session resumes instead of restarting (every experiment skips itself if its result
file exists). On Kaggle everything goes to `/kaggle/working`, the notebook's saved output.

In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    """Where results and figures are written.

    On Kaggle this must be /kaggle/working -- that directory IS the kernel's output, and
    anything written elsewhere is discarded when the session ends.
    """
    if os.path.exists("/kaggle/working"):
        return Path("/kaggle/working")
    markers = ("README.md", "requirements.txt", ".git")
    for start in (Path.cwd(), Path.cwd().parent):
        for candidate in [start, *start.parents]:
            if any((candidate / m).exists() for m in markers):
                return candidate
    return Path.cwd()


DRIVE_ROOT = None
PROJECT_ROOT = find_project_root()

if IN_COLAB and CONFIG["storage"] == "drive":
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path("/content/drive/MyDrive") / CONFIG["project"]
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        PROJECT_ROOT = DRIVE_ROOT
        print(f"Drive mounted -> {DRIVE_ROOT}")
    except Exception as exc:
        print(f"Drive mount failed ({exc}); using ephemeral /content storage.")
        PROJECT_ROOT = Path("/content") / CONFIG["project"]

if DRIVE_ROOT is not None:
    STORE_ROOT = DRIVE_ROOT
elif CONFIG["storage"] == "ephemeral":
    STORE_ROOT = Path("/content/scratch") if IN_COLAB else Path(tempfile.gettempdir()) / CONFIG["project"]
    print(f"storage=ephemeral -> gigabyte artifacts go to {STORE_ROOT} (not kept)")
else:
    STORE_ROOT = PROJECT_ROOT

PATHS = {
    # small outputs (committed to git)
    "root": PROJECT_ROOT,
    "data": PROJECT_ROOT / "data",
    "audit": PROJECT_ROOT / "data" / "audit",   # dataset audit evidence only
    "results": PROJECT_ROOT / "results",
    "runs": PROJECT_ROOT / "results" / "runs",
    "predictions": PROJECT_ROOT / "results" / "predictions",
    "figures": PROJECT_ROOT / "figures",
    # large artefacts (gitignored)
    "raw": STORE_ROOT / "data" / "raw",
    "models": STORE_ROOT / "models",
    "hf_cache": STORE_ROOT / "hf_cache",
}
EAGER = ("data", "audit", "results", "runs", "predictions", "figures")
for _name in EAGER:
    PATHS[_name].mkdir(parents=True, exist_ok=True)

# Set before transformers is imported, so model downloads go to hf_cache.
os.environ["HF_HOME"] = str(PATHS["hf_cache"])
CONFIG["paths"] = {k: str(v) for k, v in PATHS.items()}

print(f"project root : {PATHS['root']}")
print(f"large store  : {STORE_ROOT}" + ("  (Google Drive)" if DRIVE_ROOT else "  (local)"))
for _name in ("results", "figures", "models", "hf_cache"):
    print(f"  {_name:10} -> {PATHS[_name]}")
if DRIVE_ROOT is None and not IN_COLAB:
    print("\n  No Drive: model downloads and checkpoints would land on this machine.")
    print("  Section 4 refuses to fetch the full-size encoders without a GPU for that reason.")

### 0.4 Seeding

Fixes the Python, NumPy and PyTorch seeds (42) and requests deterministic cuDNN kernels, so
a run is repeatable. Repeatable is not the same as robust: Section 5.2 re-trains the
stochastic models with two more seeds to measure run-to-run variation.

In [ ]:
import random
import tempfile

import numpy as np


def set_seed(seed: int) -> None:
    """Seed every source of randomness the project touches."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


set_seed(CONFIG["seed"])
print(f"seeded everything with {CONFIG['seed']}")

### 0.5 Hardware check

Reports the training device. Mixed precision (`fp16`) is enabled only when a GPU is
present.

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG["transformer"]["fp16"] = bool(torch.cuda.is_available()) and CONFIG["transformer"]["fp16"]

print(f"torch        : {torch.__version__}")
print(f"device       : {DEVICE}")
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    print(f"gpu          : {_props.name} ({_props.total_memory / 1e9:.1f} GB)")
else:
    print(f"cpu threads  : {torch.get_num_threads()}")
    print("  No GPU: E3/E4 full fine-tuning is not practical here. Use fast_mode locally")
    print("  for a smoke test and run the real Transformer experiments on Colab.")
print(f"fp16 enabled : {CONFIG['transformer']['fp16']}")

## 1. Dataset

### 1.1 Loading KINNEWS

The Hugging Face version of KINNEWS ships as a loading script, which current `datasets`
releases no longer run. The cell tries each loading route in turn and records which one
worked; in practice the authors' release archive is read directly, and its SHA-256 hash is
printed to pin the exact version.

In [ ]:
import hashlib
import io
import json
import re
import time
import unicodedata
import urllib.request
import zipfile

import pandas as pd

RAW_DIR = PATHS["raw"]

KINNEWS_ZIP_URL = "https://github.com/saradhix/kinnews_kirnews/raw/master/KINNEWS.zip"
KINNEWS_HOMEPAGE = "https://github.com/Andrews2017/KINNEWS-and-KIRNEWS-Corpus"
LOAD_LOG = []


def _record(route, status, detail):
    LOAD_LOG.append({"route": route, "status": status, "detail": str(detail)[:200]})
    print(f"  [{status:8}] route {route}: {str(detail)[:150]}")


def _route_1_hub_script():
    from datasets import load_dataset

    return load_dataset(CONFIG["data"]["hf_id"], CONFIG["data"]["hf_config"])


def _route_2_parquet():
    from datasets import load_dataset

    return load_dataset(
        CONFIG["data"]["hf_id"], CONFIG["data"]["hf_config"], revision="refs/convert/parquet"
    )


def _route_4_authors_release():
    """Fetch the authors' archive and read the raw CSVs.

    With CONFIG["stream_corpus"], the 27 MB archive is held in memory and never written to
    disk -- so the corpus leaves no trace on the machine. Set it to False to cache the file
    and avoid re-downloading on every run.
    """
    if CONFIG["stream_corpus"]:
        req = urllib.request.Request(KINNEWS_ZIP_URL, headers={"User-Agent": "python-urllib"})
        with urllib.request.urlopen(req, timeout=300) as resp:
            blob = resp.read()
        print(f"  streamed {len(blob):,} bytes into memory (nothing written to disk)")
        source = io.BytesIO(blob)
    else:
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        zip_path = RAW_DIR / "KINNEWS.zip"
        if not zip_path.exists():
            req = urllib.request.Request(KINNEWS_ZIP_URL, headers={"User-Agent": "python-urllib"})
            with urllib.request.urlopen(req, timeout=300) as resp, open(zip_path, "wb") as fh:
                fh.write(resp.read())
        blob = zip_path.read_bytes()
        print(f"  cached archive: {zip_path} ({len(blob):,} bytes)")
        source = io.BytesIO(blob)

    digest = hashlib.sha256(blob).hexdigest()
    print(f"  sha256={digest[:16]}...  (pins the exact corpus version)")
    frames = {}
    with zipfile.ZipFile(source) as zf:
        for split in ("train", "test"):
            with zf.open(f"KINNEWS/raw/{split}.csv") as fh:
                frames[split] = pd.read_csv(io.BytesIO(fh.read()), encoding="utf-8")
        class_names = zf.read("KINNEWS/classes.txt").decode("utf-8").split()
    return frames, class_names, digest


print("Attempting the four loading routes in order:\n")

train_df = test_df = class_names = None

try:
    _ds = _route_1_hub_script()
    _record(1, "OK", "load_dataset with the Hub loading script succeeded")
    train_df, test_df = _ds["train"].to_pandas(), _ds["test"].to_pandas()
except Exception as exc:
    _record(1, "FAILED", f"{type(exc).__name__}: {exc}")

if train_df is None:
    try:
        _ds = _route_2_parquet()
        _record(2, "OK", "Hub parquet conversion succeeded")
        train_df, test_df = _ds["train"].to_pandas(), _ds["test"].to_pandas()
    except Exception as exc:
        _record(2, "FAILED", f"{type(exc).__name__}: {exc}")

# Route 3 (datasets<3.0) needs a conflicting environment and cannot run in-session.
_record(3, "SKIPPED", "would require downgrading `datasets` in-session; see DECISIONS.md D4")

if train_df is None:
    _frames, class_names, _digest = _route_4_authors_release()
    train_df, test_df = _frames["train"], _frames["test"]
    _record(4, "OK", f"authors' release parsed directly ({KINNEWS_HOMEPAGE})")

if class_names is None:
    class_names = [
        "politics", "sport", "economy", "health", "entertainment", "history", "technology",
        "tourism", "culture", "fashion", "religion", "environment", "education", "relationship",
    ]

LABELS = list(class_names)
CONFIG["data"]["class_names"] = LABELS
print(f"\nLoaded: train={train_df.shape}, test={test_df.shape}")
print(f"Classes ({len(LABELS)}): {', '.join(LABELS)}")

### 1.2 Checking the published description

Verifies row counts, number of classes, label range, field names and missing values
against the paper, with a PASS/FAIL for each.

In [ ]:
CLAIMS = {
    "train rows == 17,014": (len(train_df), 17_014),
    "test rows == 4,254": (len(test_df), 4_254),
    "classes == 14": (train_df["label"].nunique(), 14),
    "labels are 1..14": ((train_df["label"].min(), train_df["label"].max()), (1, 14)),
}
print("Published claim vs. the file on disk:\n")
for claim, (actual, expected) in CLAIMS.items():
    print(f"  {'PASS' if actual == expected else 'FAIL'}  {claim:24} actual={actual}")

expected_cols = ["label", "kin_label", "en_label", "url", "title", "content"]
print(f"\n  {'PASS' if list(train_df.columns) == expected_cols else 'FAIL'}  "
      f"columns == {expected_cols}")
print(f"\nMissing values (train): {train_df.isna().sum().to_dict()}")
print(f"Missing values (test) : {test_df.isna().sum().to_dict()}")

label_map = (
    train_df.groupby("label")[["kin_label", "en_label"]]
    .agg(lambda s: s.mode().iat[0])
    .assign(train_n=train_df["label"].value_counts().sort_index(),
            test_n=test_df["label"].value_counts().sort_index())
)
label_map["train_%"] = (label_map.train_n / len(train_df) * 100).round(2)
label_map["test_%"] = (label_map.test_n / len(test_df) * 100).round(2)
print("\nClass index -> Kinyarwanda / English label, with counts:\n")
print(label_map.to_string())

_imb = label_map.train_n.max() / label_map.train_n.min()
print(f"\nTrain imbalance ratio (largest/smallest class): {_imb:.1f}x")
print(f"  largest : {label_map.train_n.idxmax()} = {label_map.loc[label_map.train_n.idxmax(),'en_label']}"
      f" ({label_map.train_n.max():,})")
print(f"  smallest: {label_map.train_n.idxmin()} = {label_map.loc[label_map.train_n.idxmin(),'en_label']}"
      f" ({label_map.train_n.min():,})")

**Two findings from these checks.** The smallest training class is environment
(38 articles), not education (116). More importantly, the official train and test splits
**do not share a class distribution**: environment is ~19× more frequent in test, fashion
~11×, and economy ~5× less. Every test score has to be read with this in mind; 1.5
quantifies it.

### 1.3 Quality audit

Checks for exact and near duplicates, train/test leakage, very short articles and
conflicting labels, then decides what to remove. The audit runs **before** the validation
split: otherwise copies of the same article would land in both train and validation, and
validation would reward memorisation.

In [ ]:
def normalise_for_dedup(text: str) -> str:
    """Aggressive normalisation used ONLY to detect duplicates, never for modelling."""
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


for _df in (train_df, test_df):
    _df["full_text"] = _df["title"].fillna("") + " " + _df["content"].fillna("")
    _df["dedup_key"] = _df["full_text"].map(normalise_for_dedup)

audit = {}
audit["train_rows"] = len(train_df)
audit["test_rows"] = len(test_df)
audit["train_exact_dup_rows_raw"] = int(train_df["full_text"].duplicated().sum())
audit["train_exact_dup_rows_normalised"] = int(train_df["dedup_key"].duplicated().sum())
audit["train_duplicate_url_rows"] = int(train_df["url"].duplicated().sum())
audit["train_distinct_texts"] = int(train_df["dedup_key"].nunique())
audit["test_exact_dup_rows_normalised"] = int(test_df["dedup_key"].duplicated().sum())

print("Exact duplicates")
print(f"  train rows that repeat earlier text (untouched) : {audit['train_exact_dup_rows_raw']:,}")
print(f"  train rows that repeat earlier text (normalised): {audit['train_exact_dup_rows_normalised']:,}")
print(f"  train rows with a repeated URL                  : {audit['train_duplicate_url_rows']:,}")
print(f"  distinct training texts                         : {audit['train_distinct_texts']:,}"
      f" of {len(train_df):,}"
      f"  ({100*audit['train_exact_dup_rows_normalised']/len(train_df):.1f}% are copies)")
print(f"  test rows that repeat earlier text (normalised) : {audit['test_exact_dup_rows_normalised']:,}")

_grp = train_df.groupby("dedup_key")
_rep = _grp.size().sort_values(ascending=False)
print(f"\n  most-repeated single article appears {_rep.iloc[0]}x")
print(f"  title: {train_df.loc[train_df.dedup_key == _rep.index[0], 'title'].iloc[0][:100]}")

_nlab = _grp["label"].nunique()
_conflict = _nlab[_nlab > 1]
audit["dup_groups_with_conflicting_labels"] = int(len(_conflict))
audit["rows_in_label_conflicting_groups"] = int(train_df["dedup_key"].isin(_conflict.index).sum())
print(f"\nLabel noise revealed by duplicates")
print(f"  duplicate groups carrying >1 distinct label: {audit['dup_groups_with_conflicting_labels']:,}")
print(f"  training rows involved                     : {audit['rows_in_label_conflicting_groups']:,}")
print("  -> identical text labelled two different ways puts a ceiling on achievable accuracy.")

**Leakage check.** Exact train/test overlap first, then near-duplicates by TF-IDF
cosine similarity (> 0.9), which catches lightly edited republications.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

test_keys = set(test_df["dedup_key"])
leak_mask = train_df["dedup_key"].isin(test_keys)
audit["train_test_exact_overlap_rows"] = int(leak_mask.sum())
print(f"Exact train/test overlap: {audit['train_test_exact_overlap_rows']} training rows "
      f"are verbatim copies of a test article.")

_t0 = time.time()
_dedup_train = train_df.drop_duplicates("dedup_key", keep="first")
_vec = TfidfVectorizer(min_df=2, max_features=60_000, ngram_range=(1, 1))
_Xtr = normalize(_vec.fit_transform(_dedup_train["dedup_key"]))
_Xte = normalize(_vec.transform(test_df["dedup_key"]))

_near, _best = 0, []
_thr = CONFIG["audit"]["near_dup_threshold"]
for _s in range(0, _Xte.shape[0], 500):
    _sim = (_Xte[_s:_s + 500] @ _Xtr.T).toarray()
    _mx = _sim.max(axis=1)
    _near += int((_mx > _thr).sum())
    _best.append(_mx)
_best = np.concatenate(_best)
audit["train_test_near_dup_rows"] = _near
audit["near_dup_threshold"] = _thr
print(f"Near-duplicates (TF-IDF cosine > {_thr}): {_near} of {len(test_df):,} test articles "
      f"have a very similar training article  ({100*_near/len(test_df):.2f}%)")
print(f"  max similarity seen: {_best.max():.3f};  median: {np.median(_best):.3f}"
      f"   [computed in {time.time()-_t0:.1f}s]")

**Short and empty articles** are counted and reported, not dropped.

In [ ]:
train_df["n_words"] = train_df["full_text"].str.split().str.len()
test_df["n_words"] = test_df["full_text"].str.split().str.len()
_min_w = CONFIG["audit"]["min_words"]
audit["train_short_articles"] = int((train_df.n_words < _min_w).sum())
audit["test_short_articles"] = int((test_df.n_words < _min_w).sum())
audit["test_null_content"] = int(test_df["content"].isna().sum())
print(f"Articles under {_min_w} words:  train={audit['train_short_articles']}, "
      f"test={audit['test_short_articles']}")
print(f"Test rows with empty content (title only): {audit['test_null_content']}")
if audit["test_null_content"]:
    print("  -> kept: they still carry a title, and the test set is never modified.")
print(f"Shortest training article: {train_df.n_words.min()} words")

**Label spot check.** Twenty random training articles with their labels, saved for
manual review (7.5b). Judging a label needs a Kinyarwanda reader, so the code makes no
claim about them.

In [ ]:
_spot = train_df.sample(CONFIG["audit"]["label_spot_check_n"], random_state=CONFIG["seed"])
spot_check = pd.DataFrame({
    "row": _spot.index,
    "label": _spot["en_label"].values,
    "title": _spot["title"].str.slice(0, 70).values,
    "snippet": _spot["content"].astype(str).str.slice(0, 90).values,
})
spot_check.to_csv(PATHS["audit"] / "label_spot_check.csv", index=False)
print("FOR MANUAL REVIEW BY THE STUDENT — the notebook asserts nothing about these labels.\n")
print(spot_check.to_string(index=False))
print(f"\nsaved -> {PATHS['audit'] / 'label_spot_check.csv'}")

**Deduplication.** Exact duplicates within train (keeping the first copy) and training
articles copied from the test set are removed; the test set stays exactly as published.
This drops about 55% of the training rows but no information, since every removed row is
an identical copy of one that is kept.

In [ ]:
_before = len(train_df)
train_df = train_df[~leak_mask].drop_duplicates("dedup_key", keep="first").reset_index(drop=True)
audit["train_rows_after_dedup"] = len(train_df)
audit["train_rows_removed"] = _before - len(train_df)

_counts_after = train_df["label"].value_counts().sort_index()
audit["imbalance_ratio_before"] = round(float(_imb), 1)
audit["imbalance_ratio_after"] = round(float(_counts_after.max() / _counts_after.min()), 1)

print(f"train: {_before:,} -> {len(train_df):,} rows "
      f"({audit['train_rows_removed']:,} removed, {100*audit['train_rows_removed']/_before:.1f}%)")
print(f"test : {len(test_df):,} rows (untouched, as published)")
print(f"\nimbalance ratio: {audit['imbalance_ratio_before']}x -> "
      f"{audit['imbalance_ratio_after']}x after deduplication")
print("\nDuplicates removed per class (they were concentrated in the biggest classes):")
_removed = (label_map.train_n - _counts_after).fillna(0).astype(int)
print(pd.DataFrame({"class": label_map.en_label, "removed": _removed,
                    "remaining": _counts_after}).to_string())

with open(PATHS["audit"] / "audit_summary.json", "w") as fh:
    json.dump(audit, fh, indent=2)
print(f"\nsaved -> {PATHS['audit'] / 'audit_summary.json'}")

### 1.4 Splits

A stratified 10% validation split of the deduplicated training data (seed 42).
Stratifying keeps every class in validation even when it has only a few dozen articles.
The indices are saved to `data/splits.json`, so every model uses identical data.

In [ ]:
from sklearn.model_selection import train_test_split

_tr_idx, _va_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=CONFIG["data"]["val_size"],
    random_state=CONFIG["seed"],
    stratify=train_df["label"].values,
)
splits = {
    "seed": CONFIG["seed"],
    "val_size": CONFIG["data"]["val_size"],
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "note": "Indices are positions in the DEDUPLICATED train frame; test is the official split, untouched.",
    "audit": audit,
    "train_idx": sorted(int(i) for i in _tr_idx),
    "val_idx": sorted(int(i) for i in _va_idx),
    "n_test": len(test_df),
}
with open(PATHS["data"] / "splits.json", "w") as fh:
    json.dump(splits, fh, indent=2)

tr_df = train_df.iloc[splits["train_idx"]].reset_index(drop=True)
va_df = train_df.iloc[splits["val_idx"]].reset_index(drop=True)
te_df = test_df.reset_index(drop=True)

print(f"train {len(tr_df):,}   val {len(va_df):,}   test {len(te_df):,}")
print(f"saved -> {PATHS['data'] / 'splits.json'}")

assert set(tr_df.dedup_key) & set(va_df.dedup_key) == set(), "train/val overlap!"
assert set(tr_df.dedup_key) & set(te_df.dedup_key) == set(), "train/test overlap!"
assert set(va_df.dedup_key) & set(te_df.dedup_key) == set(), "val/test overlap!"
print("\nPASS  no article text is shared between train, validation and test.")

split_counts = pd.DataFrame({
    "class": label_map.en_label,
    "train": tr_df.label.value_counts().sort_index(),
    "val": va_df.label.value_counts().sort_index(),
    "test": te_df.label.value_counts().sort_index(),
}).fillna(0).astype({"train": int, "val": int, "test": int})
print("\nClass counts per split:\n")
print(split_counts.to_string())

### 1.5 Exploratory analysis

Five figures, each tied to a later decision:

| Figure | Informs |
|---|---|
| Class distribution per split | macro-F1 as the primary metric; class-weighted losses |
| Train vs test distribution shift | how every test number is interpreted |
| Length distribution (words and subword tokens) | `max_length` 512, and ablation A2 |
| Tokenizer fertility | the morphology argument behind RQ1–RQ3 |
| Vocabulary growth and hapax rate | why character n-grams and subwords should help |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
FIGDPI = 200


def save_fig(fig, name: str) -> None:
    path = PATHS["figures"] / f"{name}.png"
    fig.savefig(path, dpi=FIGDPI, bbox_inches="tight")
    print(f"saved -> {path}")


# ---- Figure 1: class distribution per split ----
fig, ax = plt.subplots(figsize=(11, 5))
_plot = split_counts.set_index("class")[["train", "val", "test"]]
_plot.plot(kind="bar", ax=ax, width=0.8)
ax.set_title(f"KINNEWS class distribution by split "
             f"(train imbalance {audit['imbalance_ratio_after']}x after deduplication)")
ax.set_xlabel("Topic")
ax.set_ylabel("Number of articles")
ax.legend(title="Split")
plt.xticks(rotation=45, ha="right")
save_fig(fig, "01_class_distribution_by_split")
plt.show()

**Figure 2** compares each class's share of the test set with its share of training
(log scale; 1.0 means equal). It is the key plot for reading every test result.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
_share = pd.DataFrame({
    "train_%": tr_df.label.value_counts(normalize=True).sort_index() * 100,
    "test_%": te_df.label.value_counts(normalize=True).sort_index() * 100,
}).fillna(0)
_share.index = label_map.en_label.values
_share["ratio"] = _share["test_%"] / _share["train_%"].replace(0, np.nan)
_colors = ["#c0392b" if r > 2 or r < 0.5 else "#7f8c8d" for r in _share["ratio"]]
ax.bar(_share.index, _share["ratio"], color=_colors)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, label="equal representation")
ax.set_yscale("log")
ax.set_title("Distribution shift: how much more common each topic is in test than in train")
ax.set_xlabel("Topic")
ax.set_ylabel("test share / train share  (log scale)")
ax.legend()
plt.xticks(rotation=45, ha="right")
save_fig(fig, "02_train_test_distribution_shift")
plt.show()
print(_share.round(2).to_string())

**Article length** in words and in XLM-R subword tokens. The share of articles over 256
and 512 tokens is what ablation A2 tests.

In [ ]:
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging

# truncation=False on purpose: the true length is what is being measured.
hf_logging.set_verbosity_error()
tok_xlmr = AutoTokenizer.from_pretrained(CONFIG["transformer"]["models"]["E3"])

_sample = tr_df["full_text"].sample(min(1500, len(tr_df)), random_state=CONFIG["seed"])
_tok_lens = np.array([len(tok_xlmr(t, truncation=False)["input_ids"]) for t in _sample])
_word_lens = tr_df["n_words"].values

over_256 = float((_tok_lens > 256).mean() * 100)
over_512 = float((_tok_lens > 512).mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(np.clip(_word_lens, 0, 1500), bins=60, color="#2980b9")
axes[0].axvline(np.median(_word_lens), color="black", linestyle="--",
                label=f"median {np.median(_word_lens):.0f}")
axes[0].set_title("Article length in words (train)")
axes[0].set_xlabel("Words (clipped at 1500)")
axes[0].set_ylabel("Articles")
axes[0].legend()

axes[1].hist(np.clip(_tok_lens, 0, 2000), bins=60, color="#27ae60")
for _cut, _c in ((256, "#e67e22"), (512, "#c0392b")):
    axes[1].axvline(_cut, color=_c, linestyle="--", label=f"max_length {_cut}")
axes[1].set_title("Article length in XLM-R subword tokens (sample)")
axes[1].set_xlabel("Subword tokens (clipped at 2000)")
axes[1].set_ylabel("Articles")
axes[1].legend()
save_fig(fig, "03_length_distributions")
plt.show()

print(f"words  : mean {_word_lens.mean():.0f}, median {np.median(_word_lens):.0f}, "
      f"max {_word_lens.max():,}")
print(f"tokens : mean {_tok_lens.mean():.0f}, median {np.median(_tok_lens):.0f}, "
      f"max {_tok_lens.max():,}")
print(f"\n{over_256:.1f}% of articles exceed 256 subword tokens")
print(f"{over_512:.1f}% of articles exceed 512 subword tokens")
print(f"-> at max_length=512 the model still never sees the tail of {over_512:.1f}% of articles.")

**Tokenizer fertility** is the number of subword tokens per word. High fertility means
the language is split into small pieces: each carries less meaning, and a 512-token
window covers less text. Measured for both XLM-R and AfroXLMR.

In [ ]:
tok_afro = AutoTokenizer.from_pretrained(CONFIG["transformer"]["models"]["E4"])

_fert_sample = tr_df["full_text"].sample(min(400, len(tr_df)), random_state=CONFIG["seed"]).tolist()
fertility = {}
for _name, _tk in (("XLM-R", tok_xlmr), ("AfroXLMR", tok_afro)):
    _n_tok = sum(len(_tk.tokenize(t)) for t in _fert_sample)
    _n_wrd = sum(len(t.split()) for t in _fert_sample)
    fertility[_name] = _n_tok / _n_wrd

identical = (
    tok_xlmr.vocab_size == tok_afro.vocab_size
    and all(tok_xlmr.tokenize(t) == tok_afro.tokenize(t) for t in _fert_sample[:50])
)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(list(fertility), list(fertility.values()), color=["#8e44ad", "#16a085"])
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, label="1 token per word (ideal)")
for i, (k, v) in enumerate(fertility.items()):
    ax.text(i, v + 0.03, f"{v:.2f}", ha="center", fontweight="bold")
ax.set_title("Tokenizer fertility on Kinyarwanda news\n(subword tokens per whitespace word)")
ax.set_ylabel("Tokens per word")
ax.legend()
save_fig(fig, "04_tokenizer_fertility")
plt.show()

for _k, _v in fertility.items():
    print(f"{_k:10}: {_v:.3f} subword tokens per word  (vocab {tok_xlmr.vocab_size:,})")
print(f"\nDo the two tokenizers produce identical output? {identical}")
print("Example word split by XLM-R:", tok_xlmr.tokenize("Umuryango w'abibumbye uratangaza"))

**The two tokenizers are identical** (same 250k vocabulary, same output, fertility
2.47). Any difference between E3 and E4 in Section 4 therefore comes from the adapted
weights, not from tokenization. The high fertility itself reflects a vocabulary built
without Kinyarwanda.

In [ ]:
# ---- Figure 5: vocabulary richness ----
_tokens = " ".join(tr_df["full_text"].str.lower()).split()
_vocab = pd.Series(_tokens).value_counts()
hapax = float((_vocab == 1).mean() * 100)
ttr = len(_vocab) / len(_tokens)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
_cum = np.cumsum(_vocab.values) / len(_tokens) * 100
axes[0].plot(np.arange(1, len(_cum) + 1), _cum, color="#c0392b")
axes[0].set_xscale("log")
axes[0].set_title("Corpus coverage by vocabulary size")
axes[0].set_xlabel("Distinct word types (log scale)")
axes[0].set_ylabel("% of all word tokens covered")
axes[0].axhline(90, linestyle="--", color="grey", linewidth=1)

axes[1].bar(["appears once\n(hapax)", "appears 2-5x", "appears >5x"],
            [(_vocab == 1).sum(), ((_vocab >= 2) & (_vocab <= 5)).sum(), (_vocab > 5).sum()],
            color=["#c0392b", "#e67e22", "#27ae60"])
axes[1].set_title(f"Word types by frequency  ({hapax:.0f}% appear exactly once)")
axes[1].set_ylabel("Number of word types")
save_fig(fig, "05_vocabulary_richness")
plt.show()

print(f"word tokens      : {len(_tokens):,}")
print(f"distinct types   : {len(_vocab):,}")
print(f"type/token ratio : {ttr:.4f}")
print(f"hapax legomena   : {hapax:.1f}% of types appear exactly once")
print("-> a word-level model must treat most of its vocabulary as near-unseen. This is the")
print("   agglutination argument behind RQ1 (character n-grams) and RQ2 (fastText subwords).")

**Top terms per class**, as a check that the labels match their content.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

_cv = CountVectorizer(min_df=3, max_features=40_000)
_X = _cv.fit_transform(tr_df["full_text"].str.lower())
_terms = np.array(_cv.get_feature_names_out())
_rows = []
for _lab in sorted(tr_df.label.unique()):
    _m = (tr_df.label == _lab).values
    _in = np.asarray(_X[_m].sum(0)).ravel() / max(_m.sum(), 1)
    _out = np.asarray(_X[~_m].sum(0)).ravel() / max((~_m).sum(), 1)
    _score = (_in + 0.5) / (_out + 0.5)
    _rows.append({"class": label_map.loc[_lab, "en_label"],
                  "distinctive terms": ", ".join(_terms[np.argsort(-_score)[:8]])})
print(pd.DataFrame(_rows).to_string(index=False))

### 1.6 Preprocessing

Two pipelines, because the two model families need different input:

- `preprocess_classical` (TF-IDF and BiLSTM): Unicode NFC, lowercase, URLs, e-mails and
  numbers replaced by placeholder tokens, punctuation removed **except apostrophes**, which
  mark elision in Kinyarwanda (`w'abibumbye`, `n'ibindi`).
- `preprocess_transformer`: NFC and whitespace only, keeping the text close to what the
  pretrained models saw.

The model input is the title followed by the article body.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
NUM_RE = re.compile(r"\b\d[\d.,]*\b")
KEEP_RE = re.compile(r"[^\w\s'’<>]", flags=re.UNICODE)


def preprocess_classical(text: str) -> str:
    """For TF-IDF and the BiLSTM: normalise hard, keep apostrophes."""
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = URL_RE.sub(f" {CONFIG['preprocess']['url_token']} ", text)
    text = EMAIL_RE.sub(f" {CONFIG['preprocess']['email_token']} ", text)
    text = NUM_RE.sub(f" {CONFIG['preprocess']['num_token']} ", text)
    text = KEEP_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def preprocess_transformer(text: str) -> str:
    """For XLM-R / AfroXLMR: leave the text looking like natural news prose."""
    text = unicodedata.normalize("NFC", str(text))
    return re.sub(r"\s+", " ", text).strip()


def build_text(df: pd.DataFrame, mode: str = None) -> pd.Series:
    """title + ' . ' + content by default; title only for ablation A1."""
    mode = mode or CONFIG["preprocess"]["text_mode"]
    title = df["title"].fillna("")
    if mode == "title_only":
        return title
    return title + " . " + df["content"].fillna("")


_demo = build_text(tr_df).iloc[0]
print("RAW           :", _demo[:150].replace("\n", " "))
print("\nCLASSICAL     :", preprocess_classical(_demo)[:150])
print("\nTRANSFORMER   :", preprocess_transformer(_demo)[:150])

for _name, _d in (("train", tr_df), ("val", va_df), ("test", te_df)):
    _d["text_classical"] = build_text(_d).map(preprocess_classical)
    _d["text_transformer"] = build_text(_d).map(preprocess_transformer)
    _d["y"] = _d["label"].astype(int) - 1          # 1..14 -> 0..13 for scikit-learn / torch
print(f"\nprepared: train={len(tr_df):,}  val={len(va_df):,}  test={len(te_df):,}")
print(f"label encoding: 1..14 -> 0..13, class_names[0] = {LABELS[0]!r}")


# Cache the prepared splits; the key includes the preprocessing code, so edits invalidate it.
import inspect

_prep_src = inspect.getsource(preprocess_classical) + inspect.getsource(preprocess_transformer)
CACHE_KEY = "prepared_" + hashlib.sha256(
    f"{CONFIG['seed']}|{CONFIG['data']['val_size']}|{CONFIG['preprocess']}|{_prep_src}".encode()
).hexdigest()[:16]
CACHE_DIR = PATHS["models"].parent / "prepared"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _cache_paths():
    return {k: CACHE_DIR / f"{CACHE_KEY}_{k}.parquet" for k in ("train", "val", "test")}


def save_prepared(frames):
    cols = ["label", "y", "title", "content", "full_text", "n_words",
            "text_classical", "text_transformer", "en_label", "dedup_key"]
    for name, path in _cache_paths().items():
        keep = [c for c in cols if c in frames[name].columns]
        frames[name][keep].to_parquet(path, index=False)
    print(f"cached prepared splits -> {CACHE_DIR}")


def load_prepared():
    paths = _cache_paths()
    if not all(p.exists() for p in paths.values()):
        return None
    out = {k: pd.read_parquet(v) for k, v in paths.items()}
    print(f"loaded prepared splits from cache ({CACHE_KEY}) — sections 3.1-3.6 skipped")
    print(f"  train {len(out['train']):,}  val {len(out['val']):,}  test {len(out['test']):,}")
    return out


save_prepared({"train": tr_df, "val": va_df, "test": te_df})

# Subsample only after the data section, which must describe the full corpus.
if CONFIG["fast_mode"]:
    _per_class = max(2, CONFIG["fast"]["n_train"] // len(LABELS))
    tr_df = pd.concat([g.sample(min(len(g), _per_class), random_state=CONFIG["seed"])
                       for _, g in tr_df.groupby("label")])
    va_df = va_df.sample(min(len(va_df), CONFIG["fast"]["n_val"]), random_state=CONFIG["seed"])
    te_df = te_df.sample(min(len(te_df), CONFIG["fast"]["n_test"]), random_state=CONFIG["seed"])
    tr_df, va_df, te_df = (d.reset_index(drop=True) for d in (tr_df, va_df, te_df))
    print(f"\nFAST MODE: modelling frames subsampled to train={len(tr_df)}, "
          f"val={len(va_df)}, test={len(te_df)} — smoke test only, never reported.")
    print("(The EDA above used the full splits, as it must.)")

## 2. Baselines

### 2.1 How experiments are recorded

Metric functions and `log_run`, which every experiment calls. Each run writes a JSON
result (settings, metrics, 95% bootstrap confidence interval, per-class scores) and a
predictions CSV; every later table, finding and figure is built from these files. An
experiment whose JSON already exists is skipped, so an interrupted run resumes.
Fast-mode runs write to a separate `_fast/` folder.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

ALL_LABELS = list(range(len(LABELS)))


# "seeds" only during Section 5.2, keeping extra-seed runs out of the main results.
RUN_SUBDIR = ""


def runs_dir():
    """Fast-mode results are quarantined so they can never be mistaken for real ones."""
    d = PATHS["runs"] / "_fast" if CONFIG["fast_mode"] else PATHS["runs"]
    if RUN_SUBDIR:
        d = d / RUN_SUBDIR
    d.mkdir(parents=True, exist_ok=True)
    return d


def preds_dir():
    d = PATHS["predictions"] / "_fast" if CONFIG["fast_mode"] else PATHS["predictions"]
    if RUN_SUBDIR:
        d = d / RUN_SUBDIR
    d.mkdir(parents=True, exist_ok=True)
    return d


def compute_metrics(y_true, y_pred) -> dict:
    """Macro-F1 is primary; accuracy and weighted-F1 are reported for contrast."""
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro",
                                   labels=ALL_LABELS, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted",
                                      labels=ALL_LABELS, zero_division=0)),
    }


def per_class_report(y_true, y_pred) -> list:
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=ALL_LABELS, zero_division=0
    )
    return [
        {"class": LABELS[i], "precision": float(p[i]), "recall": float(r[i]),
         "f1": float(f[i]), "support": int(s[i])}
        for i in ALL_LABELS
    ]


def bootstrap_macro_f1_ci(y_true, y_pred, n_resamples=None, ci=None):
    """Percentile bootstrap CI for test macro-F1 (Efron & Tibshirani, 1993).

    Resamples the test set with replacement n times and recomputes macro-F1 each time.
    It answers: if we had drawn a different test set of this size from the same source,
    how much would this score have moved? Cheap, and it stops a 1-point gap being
    reported as if it were decisive.
    """
    n_resamples = n_resamples or CONFIG["eval"]["bootstrap_n"]
    ci = ci or CONFIG["eval"]["bootstrap_ci"]
    rng = np.random.default_rng(CONFIG["seed"])
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    stats = np.empty(n_resamples)
    for b in range(n_resamples):
        idx = rng.integers(0, len(y_true), len(y_true))
        stats[b] = f1_score(y_true[idx], y_pred[idx], average="macro",
                            labels=ALL_LABELS, zero_division=0)
    lo = float(np.percentile(stats, 100 * (1 - ci) / 2))
    hi = float(np.percentile(stats, 100 * (1 + ci) / 2))
    return lo, hi


def log_run(exp_id, name, family, rq, description, hyperparams,
            y_val_true, y_val_pred, y_test_true, y_test_pred,
            test_proba=None, test_texts=None, train_seconds=None,
            n_params=None, inference_ms=None, notes=None):
    """Write one experiment's result JSON and its predictions CSV. Returns the record."""
    val_m = compute_metrics(y_val_true, y_val_pred)
    test_m = compute_metrics(y_test_true, y_test_pred)
    lo, hi = bootstrap_macro_f1_ci(y_test_true, y_test_pred)

    pred_path = preds_dir() / f"{exp_id}.csv"
    rows = {
        "id": np.arange(len(y_test_true)),
        "true": [LABELS[i] for i in y_test_true],
        "predicted": [LABELS[i] for i in y_test_pred],
        "correct": np.asarray(y_test_true) == np.asarray(y_test_pred),
    }
    if test_texts is not None:
        rows["snippet"] = [str(t)[:160].replace("\n", " ") for t in test_texts]
    if test_proba is not None:
        proba = np.asarray(test_proba)
        top3 = np.argsort(-proba, axis=1)[:, :3]
        for k in range(3):
            rows[f"top{k+1}_label"] = [LABELS[i] for i in top3[:, k]]
            rows[f"top{k+1}_prob"] = proba[np.arange(len(proba)), top3[:, k]].round(4)
    pd.DataFrame(rows).to_csv(pred_path, index=False)

    record = {
        "id": exp_id,
        "name": name,
        "family": family,
        "research_question": rq,
        "description": description,
        "fast_mode": CONFIG["fast_mode"],
        "seed": CONFIG["seed"],
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "hyperparameters": hyperparams,
        "n_train": int(len(tr_df)), "n_val": int(len(va_df)), "n_test": int(len(te_df)),
        "val_metrics": val_m,
        "test_metrics": test_m,
        "test_macro_f1_ci95": [lo, hi],
        "per_class_test": per_class_report(y_test_true, y_test_pred),
        "train_seconds": train_seconds,
        "n_parameters": n_params,
        "inference_ms_per_article": inference_ms,
        "peak_gpu_mb": (torch.cuda.max_memory_allocated() / 1e6
                        if torch.cuda.is_available() else None),
        "predictions_file": str(pred_path.relative_to(PATHS["root"])),
        "notes": notes,
    }
    out = runs_dir() / f"{exp_id}.json"
    with open(out, "w") as fh:
        json.dump(record, fh, indent=2)

    tag = "  [FAST MODE — not reportable]" if CONFIG["fast_mode"] else ""
    print(f"\n{exp_id} ({name}){tag}")
    print(f"  val  macro-F1 {val_m['macro_f1']:.4f}   acc {val_m['accuracy']:.4f}")
    print(f"  test macro-F1 {test_m['macro_f1']:.4f}  [95% CI {lo:.4f}–{hi:.4f}]"
          f"   acc {test_m['accuracy']:.4f}")
    print(f"  saved -> {out.relative_to(PATHS['root'])}  +  {pred_path.relative_to(PATHS['root'])}")
    return record


def is_done(exp_id) -> bool:
    """Resume support: an experiment whose result already exists skips itself."""
    if (runs_dir() / f"{exp_id}.json").exists():
        print(f"{exp_id}: result already exists, skipping (delete the JSON to force a rerun).")
        return True
    return False


print(f"results  -> {runs_dir()}")
print(f"preds    -> {preds_dir()}")
print(f"bootstrap: {CONFIG['eval']['bootstrap_n']} resamples, "
      f"{int(CONFIG['eval']['bootstrap_ci']*100)}% CI")

### 2.2 B0 — majority class

Always predicts the largest training class. It sets the floor and shows why macro-F1 is
the primary metric: accuracy rewards guessing the biggest class, while macro-F1 weights
all 14 classes equally.

In [ ]:
if not is_done("B0"):
    _t0 = time.time()
    majority = int(tr_df["y"].value_counts().idxmax())
    val_pred = np.full(len(va_df), majority)
    test_pred = np.full(len(te_df), majority)
    proba = np.zeros((len(te_df), len(LABELS)))
    proba[:, majority] = 1.0

    print(f"Majority class in train: {LABELS[majority]!r} "
          f"({(tr_df.y == majority).mean()*100:.1f}% of training rows)")
    log_run(
        exp_id="B0", name="Majority class", family="baseline", rq="floor",
        description="Always predicts the most frequent training class.",
        hyperparams={"majority_class": LABELS[majority]},
        y_val_true=va_df.y.values, y_val_pred=val_pred,
        y_test_true=te_df.y.values, y_test_pred=test_pred,
        test_proba=proba, test_texts=te_df["title"].values,
        train_seconds=round(time.time() - _t0, 3), n_params=1,
        notes="Floor. Demonstrates the accuracy/macro-F1 gap on imbalanced data.",
    )

### 2.3 B1 — TF-IDF word n-grams + logistic regression (the baseline)

Each article becomes a sparse vector of word and word-pair weights. TF-IDF down-weights
words found everywhere (`mu`, `na`) and boosts distinctive ones; `sublinear_tf` uses
`1 + log(count)` so repetition does not dominate. Logistic regression learns one weight per
(term, class), so the top terms per class can be printed below, and
`class_weight="balanced"` offsets the 53× class imbalance.

This is the baseline every other model must beat: strong for topic classification, trains
in seconds, needs no GPU. `C` ∈ {0.1, 1, 10} is chosen on validation only.

```
  text ─► preprocess_classical ─► TF-IDF(word 1-2 grams) ─► [n_docs x n_features] sparse
                                                                     │
                                           LogisticRegression (multinomial, balanced)
                                                                     ▼
                                                         14 class scores ─► softmax
```

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


def fit_sparse_model(kind: str, exp_id: str, name: str, rq: str, description: str):
    """Shared trainer for B1 (word n-grams) and E1 (character n-grams)."""
    cc = CONFIG["classical"]
    if kind == "word":
        vec = TfidfVectorizer(ngram_range=cc["word_ngram_range"], min_df=cc["word_min_df"],
                              max_features=cc["word_max_features"],
                              sublinear_tf=cc["sublinear_tf"])
    else:
        vec = TfidfVectorizer(analyzer=cc["char_analyzer"], ngram_range=cc["char_ngram_range"],
                              min_df=cc["char_min_df"], max_features=cc["char_max_features"],
                              sublinear_tf=cc["sublinear_tf"])

    t0 = time.time()
    Xtr = vec.fit_transform(tr_df["text_classical"])
    Xva = vec.transform(va_df["text_classical"])
    Xte = vec.transform(te_df["text_classical"])
    vec_seconds = time.time() - t0

    print(f"input  shapes: train {Xtr.shape}, val {Xva.shape}, test {Xte.shape}")
    print(f"vocabulary   : {len(vec.vocabulary_):,} features   "
          f"(density {Xtr.nnz / (Xtr.shape[0]*Xtr.shape[1]) * 100:.3f}%)")
    print(f"vectorised in {vec_seconds:.1f}s\n")

    # ---- select C on validation ----
    best = {"macro_f1": -1}
    for C in cc["C_grid"]:
        clf = LogisticRegression(C=C, max_iter=cc["max_iter"],
                                 class_weight=cc["class_weight"])
        clf.fit(Xtr, tr_df["y"])
        m = compute_metrics(va_df["y"], clf.predict(Xva))
        print(f"  C={C:<5} val macro-F1 {m['macro_f1']:.4f}  acc {m['accuracy']:.4f}")
        if m["macro_f1"] > best["macro_f1"]:
            best = {"C": C, "macro_f1": m["macro_f1"], "clf": clf}
    clf = best["clf"]
    print(f"  -> selected C={best['C']} on validation\n")

    train_seconds = time.time() - t0
    t1 = time.time()
    test_pred = clf.predict(Xte)
    inference_ms = (time.time() - t1) / len(te_df) * 1000

    print(f"output shapes: coef_ {clf.coef_.shape}  ->  {len(LABELS)} class scores per article")
    n_params = int(clf.coef_.size + clf.intercept_.size)
    print(f"parameters   : {n_params:,} (all trainable)")

    rec = log_run(
        exp_id=exp_id, name=name, family="classical", rq=rq, description=description,
        hyperparams={"C": best["C"], "C_grid": cc["C_grid"], "class_weight": cc["class_weight"],
                     "ngram_range": list(vec.ngram_range), "analyzer": vec.analyzer,
                     "n_features": len(vec.vocabulary_), "sublinear_tf": cc["sublinear_tf"]},
        y_val_true=va_df["y"].values, y_val_pred=clf.predict(Xva),
        y_test_true=te_df["y"].values, y_test_pred=test_pred,
        test_proba=clf.predict_proba(Xte), test_texts=te_df["title"].values,
        train_seconds=round(train_seconds, 2), n_params=n_params,
        inference_ms=round(inference_ms, 3),
    )
    return vec, clf, rec


def show_top_features(vec, clf, n=None, classes=None):
    """Print the highest-weighted features per class — the model, made readable."""
    n = n or CONFIG["classical"]["top_features_per_class"]
    names = np.array(vec.get_feature_names_out())
    classes = classes if classes is not None else range(len(LABELS))
    for i in classes:
        top = names[np.argsort(-clf.coef_[i])[:n]]
        print(f"  {LABELS[i]:14} {', '.join(repr(t) for t in top)}")


if not is_done("B1"):
    vec_b1, clf_b1, rec_b1 = fit_sparse_model(
        "word", "B1", "TF-IDF word 1-2 grams + LR", "baseline",
        "Sparse word n-gram features with multinomial logistic regression.",
    )
    print("\nHighest-weighted WORD features per class:")
    show_top_features(vec_b1, clf_b1)

## 3. Classical and recurrent experiments

### 3.1 E1 — TF-IDF character n-grams + logistic regression (RQ1)

Kinyarwanda builds words from prefixes and affixes around a stem, so one concept appears
in many surface forms, and 61.6% of word types occur only once. Character 2–5-grams
(`char_wb`, within word boundaries) let those forms share features. Everything else is
identical to B1, so any difference comes from the features.

```
  text ─► preprocess_classical ─► TF-IDF(char_wb 2-5 grams) ─► sparse matrix ─► LR ─► 14
```

In [ ]:
if not is_done("E1"):
    vec_e1, clf_e1, rec_e1 = fit_sparse_model(
        "char", "E1", "TF-IDF char 2-5 grams + LR", "RQ1",
        "Character n-gram features (char_wb) with multinomial logistic regression.",
    )
    print("\nHighest-weighted CHARACTER features per class (note the stems and affixes):")
    show_top_features(vec_e1, clf_e1)

### 3.2 Results table

Reads every result JSON into `results/experiments.csv` and `results/tables.md`; the table
grows as experiments are added.

In [ ]:
def build_results_table(write=True):
    """Assemble every result JSON into a dataframe, a CSV and a markdown table."""
    records = []
    for path in sorted(runs_dir().glob("*.json")):
        with open(path) as fh:
            d = json.load(fh)
        lo, hi = d["test_macro_f1_ci95"]
        records.append({
            "id": d["id"], "model": d["name"], "family": d["family"],
            "RQ": d["research_question"],
            "val_macro_f1": round(d["val_metrics"]["macro_f1"], 4),
            "test_macro_f1": round(d["test_metrics"]["macro_f1"], 4),
            "test_ci95_low": round(lo, 4), "test_ci95_high": round(hi, 4),
            "test_accuracy": round(d["test_metrics"]["accuracy"], 4),
            "test_weighted_f1": round(d["test_metrics"]["weighted_f1"], 4),
            "params": d["n_parameters"], "train_s": d["train_seconds"],
            "infer_ms": d["inference_ms_per_article"], "fast_mode": d["fast_mode"],
        })
    if not records:
        print("no results yet")
        return pd.DataFrame()

    df = pd.DataFrame(records).sort_values("test_macro_f1", ascending=False)
    suffix = "_FAST" if CONFIG["fast_mode"] else ""
    if write:
        csv_path = PATHS["results"] / f"experiments{suffix}.csv"
        df.to_csv(csv_path, index=False)

        lines = ["# Generated tables", "",
                 f"*Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} from "
                 f"`results/runs/`. Do not edit by hand — rerun the notebook instead.*", ""]
        if CONFIG["fast_mode"]:
            lines += ["> **FAST MODE — these numbers are a smoke test on subsampled data "
                      "and must not be reported.**", ""]
        lines += ["## Table 1. Experiment results", "",
                  "| ID | Model | RQ | Val macro-F1 | Test macro-F1 | 95% CI | Test acc. | Params |",
                  "|---|---|---|---|---|---|---|---|"]
        for _, r in df.iterrows():
            lines.append(
                f"| {r['id']} | {r['model']} | {r['RQ']} | {r['val_macro_f1']:.4f} | "
                f"**{r['test_macro_f1']:.4f}** | {r['test_ci95_low']:.3f}–{r['test_ci95_high']:.3f} | "
                f"{r['test_accuracy']:.4f} | {r['params']:,} |")

        best_id = df.iloc[0]["id"]
        with open(runs_dir() / f"{best_id}.json") as fh:
            best = json.load(fh)
        lines += ["", f"## Table 2. Per-class test metrics — best model ({best['name']})", "",
                  "| Class | Precision | Recall | F1 | Support |", "|---|---|---|---|---|"]
        for c in best["per_class_test"]:
            lines.append(f"| {c['class']} | {c['precision']:.3f} | {c['recall']:.3f} | "
                         f"{c['f1']:.3f} | {c['support']} |")

        (PATHS["results"] / f"tables{suffix}.md").write_text("\n".join(lines) + "\n")
        print(f"saved -> {csv_path.relative_to(PATHS['root'])}")
        print(f"saved -> {(PATHS['results'] / f'tables{suffix}.md').relative_to(PATHS['root'])}")
    return df


results_so_far = build_results_table()
print()
print(results_so_far[["id", "model", "val_macro_f1", "test_macro_f1",
                      "test_accuracy", "train_s"]].to_string(index=False))

### 3.2b Generated findings

`explain(a, b)` compares two runs from their result files: headline scores, whether the
confidence intervals overlap, McNemar's test, and the classes that moved most. Every "What
we learned" section uses it, so the text always matches the numbers.

In [ ]:
from IPython.display import Markdown, display


from scipy import stats


def load_preds(exp_id):
    return pd.read_csv(preds_dir() / f"{exp_id}.csv")


def mcnemar(id_a, id_b):
    pa, pb = load_preds(id_a), load_preds(id_b)
    a_ok, b_ok = pa["correct"].values.astype(bool), pb["correct"].values.astype(bool)
    b_only = int((~a_ok & b_ok).sum())      # only B right
    a_only = int((a_ok & ~b_ok).sum())      # only A right
    n = a_only + b_only
    if n == 0:
        return {"comparison": f"{id_a} vs {id_b}", "note": "models agree everywhere"}
    if n < 25:
        p = float(stats.binomtest(b_only, n, 0.5).pvalue)
        stat, kind = None, "exact binomial"
    else:
        stat = (abs(b_only - a_only) - 1) ** 2 / n
        p = float(stats.chi2.sf(stat, 1))
        kind = "chi-squared w/ continuity correction"
    return {"comparison": f"{id_a} vs {id_b}", "only_A_right": a_only, "only_B_right": b_only,
            "statistic": stat, "p_value": p, "test": kind,
            "significant_at_0.05": bool(p < 0.05)}


def _verdict(a, b):
    """Rule-based reading of two runs. Deliberately cautious."""
    lo_a, hi_a = a["test_macro_f1_ci95"]
    lo_b, hi_b = b["test_macro_f1_ci95"]
    delta = b["test_metrics"]["macro_f1"] - a["test_metrics"]["macro_f1"]
    overlap = hi_a > lo_b and hi_b > lo_a
    if abs(delta) < 0.005:
        return delta, overlap, "the two are effectively tied"
    winner = b["id"] if delta > 0 else a["id"]
    if overlap:
        return delta, overlap, (f"{winner} is ahead by {abs(delta):.4f}, but the confidence "
                                f"intervals overlap, so this is not conclusive on one seed")
    return delta, overlap, (f"{winner} is ahead by {abs(delta):.4f} and the confidence "
                            f"intervals do not overlap, so the difference survives resampling")


def explain(id_a, id_b, question, hypothesis):
    """Emit a findings section for two experiments, entirely from their result files."""
    for i in (id_a, id_b):
        if not (runs_dir() / f"{i}.json").exists():
            display(Markdown(f"> **{question}** — `{i}` has not been run yet."))
            return
    a = json.load(open(runs_dir() / f"{id_a}.json"))
    b = json.load(open(runs_dir() / f"{id_b}.json"))
    delta, overlap, verdict = _verdict(a, b)

    pa = {c["class"]: c for c in a["per_class_test"]}
    pb = {c["class"]: c for c in b["per_class_test"]}
    movers = sorted(((pb[k]["f1"] - pa[k]["f1"], k) for k in pa), key=lambda t: -abs(t[0]))[:4]
    train_n = {LABELS[i]: int((tr_df.y == i).sum()) for i in range(len(LABELS))}

    mc = mcnemar(id_a, id_b)
    mc_line = (f"McNemar: p = {mc['p_value']:.2e} — "
               f"{'a systematic difference' if mc.get('significant_at_0.05') else 'not significant'}"
               if "p_value" in mc else "McNemar: models agree everywhere")

    rows = "\n".join(
        f"| {k} | {train_n.get(k, 0):,} | {pa[k]['f1']:.3f} | {pb[k]['f1']:.3f} | {d:+.3f} |"
        for d, k in movers)

    md = f"""**{question}**

| | {a['name']} | {b['name']} |
|---|---|---|
| Validation macro-F1 | {a['val_metrics']['macro_f1']:.4f} | {b['val_metrics']['macro_f1']:.4f} |
| **Test macro-F1** | **{a['test_metrics']['macro_f1']:.4f}** | **{b['test_metrics']['macro_f1']:.4f}** |
| 95% CI | {a['test_macro_f1_ci95'][0]:.4f}–{a['test_macro_f1_ci95'][1]:.4f} | {b['test_macro_f1_ci95'][0]:.4f}–{b['test_macro_f1_ci95'][1]:.4f} |
| Test accuracy | {a['test_metrics']['accuracy']:.4f} | {b['test_metrics']['accuracy']:.4f} |
| Parameters | {a['n_parameters']:,} | {b['n_parameters']:,} |
| Training time | {a['train_seconds']}s | {b['train_seconds']}s |

**Verdict:** {verdict}. {mc_line}.

**Classes that moved most** (training-set size included, because on this corpus the gains
tend to concentrate where data is scarcest):

| Class | Train articles | {id_a} F1 | {id_b} F1 | Delta |
|---|---|---|---|---|
{rows}

**The hypothesis was:** {hypothesis}

**Validation-to-test drop:** {a['id']} loses {a['val_metrics']['macro_f1'] - a['test_metrics']['macro_f1']:+.4f}, \
{b['id']} loses {b['val_metrics']['macro_f1'] - b['test_metrics']['macro_f1']:+.4f}. Validation shares the training \
class distribution and the official test split does not (Section 1.5), so some drop is expected for every model here.
"""
    display(Markdown(md))

### 3.3 What we learned from the classical models (RQ1)

The cell below compares B1 and E1. What to look for:

- **Macro-F1 vs accuracy.** B0's accuracy is several times its macro-F1, and for B1 the `C`
  with the best accuracy is not the one with the best macro-F1: the metric changes which
  model is selected.
- **Where E1's gain comes from.** It concentrates in the smallest training classes
  (environment, tourism, fashion, culture), where sharing evidence across inflected forms
  matters most. E1's top features are stems and affixes, e.g. `dukik` from *ibidukikije*
  ("environment").
- **Scraping artefacts.** `caption` ranks among B1's top `education` terms: WordPress
  markup, not topic vocabulary.
- **Validation vs test.** Both models lose 11–13 points on test, because validation follows
  the training distribution and the test set does not.

In [ ]:
def compare(a_id, b_id):
    """Print a side-by-side of two runs, so the narrative below is written from output."""
    out = {}
    for _id in (a_id, b_id):
        with open(runs_dir() / f"{_id}.json") as fh:
            out[_id] = json.load(fh)
    a, b = out[a_id], out[b_id]
    print(f"{'':22}{a_id:>12}{b_id:>12}   delta")
    for key, fmt in (("macro_f1", "{:.4f}"), ("accuracy", "{:.4f}"), ("weighted_f1", "{:.4f}")):
        av, bv = a["test_metrics"][key], b["test_metrics"][key]
        print(f"  test {key:16}" + fmt.format(av).rjust(12) + fmt.format(bv).rjust(12)
              + f"   {bv-av:+.4f}")
    print(f"  {'95% CI':21}"
          f"{a['test_macro_f1_ci95'][0]:.3f}-{a['test_macro_f1_ci95'][1]:.3f}".rjust(12)
          + f"  {b['test_macro_f1_ci95'][0]:.3f}-{b['test_macro_f1_ci95'][1]:.3f}".rjust(12))
    print(f"  {'CIs overlap?':21}"
          f"{'yes — difference is not clearly meaningful' if a['test_macro_f1_ci95'][1] > b['test_macro_f1_ci95'][0] and b['test_macro_f1_ci95'][1] > a['test_macro_f1_ci95'][0] else 'no — difference survives resampling'}")
    print("\n  Per-class test F1 (biggest movers):")
    pa = {c["class"]: c["f1"] for c in a["per_class_test"]}
    pb = {c["class"]: c["f1"] for c in b["per_class_test"]}
    sup = {c["class"]: c["support"] for c in a["per_class_test"]}
    deltas = sorted(((pb[k] - pa[k], k) for k in pa), key=lambda t: -abs(t[0]))
    for d, k in deltas[:6]:
        print(f"    {k:14} {pa[k]:.3f} -> {pb[k]:.3f}  ({d:+.3f})   n={sup[k]}")


compare("B1", "E1")

### 3.4 E2 — a BiLSTM written from scratch (RQ2)

RQ2 asks whether subword-aware **fastText embeddings** help a neural model the way
character n-grams helped E1. The BiLSTM adds word order but no pretraining, which separates
the effect of sequence modelling from that of pretraining (E3/E4 have both).

```
  token ids  [batch, 400]
       │
  Embedding(vocab, 300)                  <- E2a: random | E2b: fastText
       │      [batch, 400, 300]
  BiLSTM(hidden=128, bidirectional)      <- reads left-to-right AND right-to-left
       │      [batch, 400, 256]             (128 forward + 128 backward, concatenated)
  masked mean-pool over real tokens      <- padding excluded from the average
       │      [batch, 256]
  Dropout(0.3) -> Linear(256, 14)
       │      [batch, 14]  class scores
```

Masked mean pooling averages over real tokens only, so padding does not pull short
articles towards the same vector. Reading in both directions gives every position left
and right context (Schuster & Paliwal, 1997).

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

PAD, UNK = 0, 1


def build_vocab(texts, min_freq):
    """Vocabulary from the TRAINING SPLIT ONLY — never validation or test."""
    from collections import Counter
    counts = Counter(w for t in texts for w in t.split())
    itos = ["<pad>", "<unk>"] + sorted(w for w, c in counts.items() if c >= min_freq)
    return {w: i for i, w in enumerate(itos)}, itos


class NewsDataset(Dataset):
    """Turns text into a fixed-length id tensor plus a padding mask."""

    def __init__(self, texts, labels, stoi, max_tokens):
        self.texts, self.labels = list(texts), list(labels)
        self.stoi, self.max_tokens = stoi, max_tokens

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        ids = [self.stoi.get(w, UNK) for w in self.texts[i].split()][: self.max_tokens]
        mask = [1] * len(ids) + [0] * (self.max_tokens - len(ids))
        ids = ids + [PAD] * (self.max_tokens - len(ids))
        return (torch.tensor(ids), torch.tensor(mask), torch.tensor(self.labels[i]))


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, n_classes, pretrained=None):
        super().__init__()
        cfg = CONFIG["bilstm"]
        self.embedding = nn.Embedding(vocab_size, cfg["embed_dim"], padding_idx=PAD)
        if pretrained is not None:                      # E2b: start from fastText
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained))
        self.lstm = nn.LSTM(cfg["embed_dim"], cfg["hidden_dim"], num_layers=cfg["num_layers"],
                            bidirectional=cfg["bidirectional"], batch_first=True)
        self.dropout = nn.Dropout(cfg["dropout"])
        self.fc = nn.Linear(cfg["hidden_dim"] * 2, n_classes)

    def forward(self, ids, mask):
        emb = self.embedding(ids)                       # [B, T, 300]
        out, _ = self.lstm(emb)                         # [B, T, 256]
        m = mask.unsqueeze(-1).float()
        pooled = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)   # masked mean -> [B, 256]
        return self.fc(self.dropout(pooled))


def run_epoch(model, loader, loss_fn, optimiser=None):
    """One pass. Training when an optimiser is given, evaluation otherwise."""
    train = optimiser is not None
    model.train() if train else model.eval()
    total, preds, gold = 0.0, [], []
    with torch.set_grad_enabled(train):
        for ids, mask, y in loader:
            ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            logits = model(ids, mask)
            loss = loss_fn(logits, y)
            if train:
                optimiser.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["bilstm"]["grad_clip"])
                optimiser.step()
            total += loss.item() * len(y)
            preds += logits.argmax(1).cpu().tolist()
            gold += y.cpu().tolist()
    return total / len(gold), compute_metrics(gold, preds), preds


def train_bilstm(exp_id, name, description, pretrained=None, notes=None):
    """Train one BiLSTM, early-stopping on validation macro-F1."""
    cfg = CONFIG["bilstm"]
    set_seed(CONFIG["seed"])
    stoi, itos = build_vocab(tr_df["text_classical"], cfg["min_freq"])
    print(f"vocabulary (train only, min_freq={cfg['min_freq']}): {len(stoi):,} types")

    loaders = {
        split: DataLoader(NewsDataset(d["text_classical"], d["y"], stoi, cfg["max_tokens"]),
                          batch_size=cfg["batch_size"], shuffle=(split == "train"))
        for split, d in (("train", tr_df), ("val", va_df), ("test", te_df))
    }
    model = BiLSTMClassifier(len(stoi), len(LABELS), pretrained).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"parameters: {n_params:,} total, {n_train:,} trainable "
          f"({model.embedding.weight.numel():,} of them in the embedding)")

    _ids, _mask, _y = next(iter(loaders["train"]))
    print(f"batch shapes -> ids {tuple(_ids.shape)}, mask {tuple(_mask.shape)}, "
          f"logits {tuple(model(_ids.to(DEVICE), _mask.to(DEVICE)).shape)}")

    weights = torch.tensor(
        len(tr_df) / (len(LABELS) * np.bincount(tr_df["y"], minlength=len(LABELS)).clip(min=1)),
        dtype=torch.float, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=weights if cfg["class_weighted_loss"] else None)
    optimiser = torch.optim.Adam(model.parameters(), lr=cfg["lr"])

    best, best_state, patience, history = -1.0, None, 0, []
    t0 = time.time()
    for epoch in range(1, cfg["epochs"] + 1):
        tr_loss, tr_m, _ = run_epoch(model, loaders["train"], loss_fn, optimiser)
        va_loss, va_m, _ = run_epoch(model, loaders["val"], loss_fn)
        history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
                        "val_macro_f1": va_m["macro_f1"]})
        flag = ""
        if va_m["macro_f1"] > best:
            best, best_state, patience = va_m["macro_f1"], \
                {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
            flag = "  <- best"
        else:
            patience += 1
        print(f"  epoch {epoch:2d}  train_loss {tr_loss:.4f}  val_loss {va_loss:.4f}  "
              f"val macro-F1 {va_m['macro_f1']:.4f}{flag}")
        if patience >= cfg["patience"]:
            print(f"  early stop: no improvement for {cfg['patience']} epochs")
            break
    train_seconds = time.time() - t0

    model.load_state_dict(best_state)                   # always report the BEST checkpoint
    _, va_m, va_pred = run_epoch(model, loaders["val"], loss_fn)
    t1 = time.time()
    _, te_m, te_pred = run_epoch(model, loaders["test"], loss_fn)
    inference_ms = (time.time() - t1) / len(te_df) * 1000

    model.eval()
    with torch.no_grad():
        proba = torch.cat([torch.softmax(model(i.to(DEVICE), m.to(DEVICE)), 1).cpu()
                           for i, m, _ in loaders["test"]]).numpy()

    rec = log_run(
        exp_id=exp_id, name=name, family="recurrent", rq="RQ2", description=description,
        hyperparams={**{k: v for k, v in cfg.items()}, "vocab_size": len(stoi),
                     "embeddings": "fastText" if pretrained is not None else "random",
                     "epochs_run": len(history), "best_val_macro_f1": best},
        y_val_true=va_df["y"].values, y_val_pred=va_pred,
        y_test_true=te_df["y"].values, y_test_pred=te_pred,
        test_proba=proba, test_texts=te_df["title"].values,
        train_seconds=round(train_seconds, 1), n_params=n_params,
        inference_ms=round(inference_ms, 3), notes=notes,
    )
    rec["history"] = history
    with open(runs_dir() / f"{exp_id}.json", "w") as fh:
        json.dump(rec, fh, indent=2)
    return model, stoi, itos, rec


print(f"device: {DEVICE}   BiLSTM config: {CONFIG['bilstm']}")

#### E2a — random embeddings

The control: every word vector is learned from the 6,863 training articles. Architecture,
data, seed and schedule are identical to E2b.

In [ ]:
if not is_done("E2a"):
    model_e2a, stoi_e2a, itos_e2a, rec_e2a = train_bilstm(
        "E2a", "BiLSTM (random embeddings)",
        "BiLSTM over word ids, embeddings initialised randomly and learned from scratch.",
        pretrained=None,
        notes="Control for RQ2: isolates the contribution of pretrained subword embeddings.",
    )

#### fastText embeddings (training split only)

fastText builds each word vector from its character 3–6-grams (Bojanowski et al., 2017),
so inflected forms share structure and unseen words still get a vector. It is trained on
the training split only, so no validation or test text leaks in, and on one thread so the
vectors are reproducible: the fingerprint printed below should match across runs.

In [ ]:
from gensim.models import FastText

_ft_tag = "fast" if CONFIG["fast_mode"] else "full"
# Worker count in the file name: never reuse a multi-threaded (non-reproducible) model.
_ft_path = PATHS["models"] / f"fasttext_train_only_{_ft_tag}_w{CONFIG['fasttext']['workers']}.model"
_fc = CONFIG["fasttext"]
if _ft_path.exists():
    ft = FastText.load(str(_ft_path))
    print(f"loaded cached fastText from {_ft_path.name}")
else:
    _t0 = time.time()
    _sentences = [t.split() for t in tr_df["text_classical"]]
    ft = FastText(vector_size=_fc["vector_size"], sg=_fc["sg"], window=_fc["window"],
                  min_count=_fc["min_count"], negative=_fc["negative"],
                  min_n=_fc["min_n"], max_n=_fc["max_n"], bucket=_fc["bucket"],
                  workers=_fc["workers"], seed=CONFIG["seed"])
    ft.build_vocab(_sentences)
    ft.train(_sentences, total_examples=len(_sentences), epochs=_fc["epochs"])
    print(f"trained fastText on {len(_sentences):,} training articles "
          f"in {time.time()-_t0:.1f}s")
    _ft_path.parent.mkdir(parents=True, exist_ok=True)
    ft.save(str(_ft_path))

print(f"fastText vocabulary: {len(ft.wv):,} words, {_fc['vector_size']}-dim, "
      f"char n-grams {_fc['min_n']}-{_fc['max_n']}")
# Identical fingerprints across runs mean identical embeddings.
FASTTEXT_FINGERPRINT = hashlib.sha256(np.ascontiguousarray(ft.wv.vectors).tobytes()).hexdigest()[:16]
print(f"fastText vector fingerprint: {FASTTEXT_FINGERPRINT}  (workers={_fc['workers']})")

**Nearest neighbours**: a quick check that the vectors group related Kinyarwanda
words.

In [ ]:
_probe = ["umukino", "imana", "ubuzima", "ishuri", "ibidukikije"]
_gloss = {"umukino": "game/match", "imana": "God", "ubuzima": "health/life",
          "ishuri": "school", "ibidukikije": "environment"}
for _w in _probe:
    try:
        _nn = ft.wv.most_similar(_w, topn=6)
        print(f"{_w:14} ({_gloss[_w]:12}) -> " +
              ", ".join(f"{n}({s:.2f})" for n, s in _nn))
    except KeyError:
        print(f"{_w:14} not in vocabulary")

_oov = "ibidukikijebyacu"          # a plausible but unattested inflected form
print(f"\nOOV check — '{_oov}' is in vocab: {_oov in ft.wv.key_to_index}")
print(f"  but still has a {ft.wv[_oov].shape[0]}-dim vector built from its character n-grams,")
print(f"  nearest: {', '.join(n for n, _ in ft.wv.most_similar(_oov, topn=5))}")
print("  This is the property Word2Vec lacks and the reason fastText was chosen.")

#### E2b — fastText initialisation

The same BiLSTM, with the embedding layer initialised from fastText and fine-tuned during
training.

In [ ]:
if not is_done("E2b"):
    _stoi, _itos = build_vocab(tr_df["text_classical"], CONFIG["bilstm"]["min_freq"])
    _matrix = np.random.normal(0, 0.1, (len(_stoi), CONFIG["bilstm"]["embed_dim"])).astype("float32")
    _matrix[PAD] = 0.0
    _hits = 0
    for _w, _i in _stoi.items():
        if _i in (PAD, UNK):
            continue
        try:
            _matrix[_i] = ft.wv[_w]        # works for OOV too, via character n-grams
            _hits += 1
        except KeyError:
            pass
    print(f"embedding matrix {_matrix.shape}: {_hits:,}/{len(_stoi)-2:,} words "
          f"({100*_hits/(len(_stoi)-2):.1f}%) initialised from fastText")

    model_e2b, stoi_e2b, itos_e2b, rec_e2b = train_bilstm(
        "E2b", "BiLSTM (fastText embeddings)",
        "Same BiLSTM, embeddings initialised from fastText skip-gram trained on train split.",
        pretrained=_matrix,
        notes="RQ2 treatment: identical to E2a except embedding initialisation.",
    )

### 3.5 What we learned from the BiLSTMs (RQ2)

Generated from `E2a.json` and `E2b.json`. Compare it with E1: the BiLSTM has 22M
parameters to fit from 6,863 articles, so landing below the character n-gram model would
be a finding about data size, not a failure.

In [ ]:
explain("E2a", "E2b",
        "RQ2: do fastText subword embeddings beat random initialisation?",
        "fastText composes word vectors from character n-grams, so morphologically related "
        "forms share structure and unseen words still get a meaningful vector. With 61.6% of "
        "word types appearing exactly once, that should matter — and by the same mechanism "
        "that made E1 beat B1, it should help the rare classes most.")
print()
_ = build_results_table()

## 4. Transformer experiments (E3, E4)

### 4.1 Fine-tuning a pretrained encoder

```
  "Umukino wa Rayon Sports ..."
       │  SentencePiece tokenizer (250,002-piece vocabulary)
  input_ids [batch, 512] + attention_mask [batch, 512]
       │  embedding lookup + positional information
  12 Transformer encoder layers, 12 self-attention heads, hidden size 768
       │  every token attends to every other token at every layer
  final hidden states [batch, 512, 768]
       │  take position 0 — the <s> classification token
  pooled [batch, 768] -> dropout -> Linear(768, 14)
       │
  logits [batch, 14] -> softmax -> probabilities;  trained with cross-entropy
```

All parameters are updated, plus a new 768×14 classification head. The learning rate is
small (2e-5; Devlin et al., 2019) so pretrained knowledge is not overwritten, with a 10%
warm-up while the new head settles. The loss is class-weighted, as for every other model.

### 4.2 The two models

| | E3 `xlm-roberta-base` | E4 `Davlan/afro-xlmr-base` |
|---|---|---|
| Pretraining | CommonCrawl (CC-100), 100 languages | XLM-R, then adapted to 17 African languages |
| Method | Masked language modelling | Multilingual adaptive fine-tuning (Alabi et al., 2022) |
| Tokenizer | SentencePiece, 250,002 pieces | **identical to XLM-R** |

Two facts make the comparison clean: the tokenizers are **identical** (1.5), and **XLM-R
never saw Kinyarwanda**: `rw` is absent from CC-100's 116 languages and from the model
card, while AfroXLMR's card lists it. RQ3 therefore asks whether adaptive fine-tuning can
add a language the base model never saw.

**The training function.** `train_transformer` tokenizes the text, fine-tunes the
encoder with the class-weighted loss (`WeightedTrainer`), keeps the best validation
checkpoint, and logs the run. E3, E4, both ablations and the seed-variance runs all call
it, so they differ only in the arguments passed.

In [ ]:
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

XLMR_CC100_CODES_SAMPLE = ["af", "am", "ha", "ig", "mg", "om", "so", "sw", "xh", "yo", "zu"]
XLMR_HAS_RW = False          # verified: "rw" absent from CC-100 and from the XLM-R card
AFROXLMR_CODES = ["en", "fr", "ar", "ha", "ig", "yo", "rn", "rw", "sn", "xh", "zu",
                  "om", "am", "so", "st", "ny", "mg", "sw", "af"]   # from the model card
AFROXLMR_HAS_RW = "rw" in AFROXLMR_CODES

print(f"Kinyarwanda ('rw') in XLM-R pretraining (CC-100) : {XLMR_HAS_RW}")
print(f"Kinyarwanda ('rw') in AfroXLMR adaptation set    : {AFROXLMR_HAS_RW}")
print(f"African languages XLM-R DID see: {', '.join(XLMR_CC100_CODES_SAMPLE)}")
print()
print("-> E3 is a true zero-pretraining control for Kinyarwanda: it brings general")
print("   multilingual structure and transfer from related Bantu languages (sw, zu, xh),")
print("   but no Kinyarwanda text. E4 was explicitly adapted to it.")
print("-> RQ3 is therefore: can adaptive fine-tuning add a language the base model never")
print("   saw, while reusing a tokenizer that was built without it?")


class HFDataset(torch.utils.data.Dataset):
    """Wraps tokenized encodings so the HF Trainer can consume them."""

    def __init__(self, encodings, labels):
        self.encodings, self.labels = encodings, list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item


def hf_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    m = compute_metrics(labels, preds)
    return {"macro_f1": m["macro_f1"], "accuracy": m["accuracy"]}


def class_weights_tensor():
    """Balanced class weights, n / (K * count_k) — the same formula the BiLSTM uses."""
    counts = np.bincount(tr_df["y"], minlength=len(LABELS)).clip(min=1)
    return torch.tensor(len(tr_df) / (len(LABELS) * counts), dtype=torch.float)


class WeightedTrainer(Trainer):
    """Trainer whose loss is class-weighted cross-entropy instead of the unweighted default."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        # logits are float16 under mixed precision; compute the loss in float32.
        loss = nn.functional.cross_entropy(logits.float(), labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


def train_transformer(exp_id, model_id, name, rq, description,
                      text_mode=None, max_length=None, notes=None):
    """Fine-tune one encoder and log it. Shared by E3, E4 and both ablations."""
    tcfg = CONFIG["transformer"]
    max_length = max_length or tcfg["max_length"]
    set_seed(CONFIG["seed"])

    tok = AutoTokenizer.from_pretrained(model_id)
    frames = {"train": tr_df, "val": va_df, "test": te_df}
    texts = {k: (build_text(d, text_mode).map(preprocess_transformer).tolist()
                 if text_mode else d["text_transformer"].tolist())
             for k, d in frames.items()}

    # Cache key includes a tokenizer fingerprint; E4 reuses E3's encodings (same tokenizer).
    _tok_fp = hashlib.sha256(
        f"{tok.__class__.__name__}|{tok.vocab_size}|"
        f"{tok.convert_tokens_to_ids(tok.tokenize('Umuryango wabibumbye'))}".encode()
    ).hexdigest()[:12]
    tok_key = (text_mode or CONFIG["preprocess"]["text_mode"], max_length, _tok_fp)
    if "_ENC_CACHE" not in globals():
        globals()["_ENC_CACHE"] = {}
    if tok_key in _ENC_CACHE:
        enc = _ENC_CACHE[tok_key]
        print(f"  tokenisation: reused from cache {tok_key}")
    else:
        _t0 = time.time()
        enc = {k: dict(tok(v, truncation=True, max_length=max_length, padding=False))
               for k, v in texts.items()}
        _ENC_CACHE[tok_key] = enc
        print(f"  tokenisation: {time.time() - _t0:.0f}s (cached for reuse)")
    datasets = {k: HFDataset(enc[k], frames[k]["y"]) for k in frames}

    cfg = AutoConfig.from_pretrained(model_id, num_labels=len(LABELS),
                                     id2label={i: l for i, l in enumerate(LABELS)},
                                     label2id={l: i for i, l in enumerate(LABELS)})
    model = AutoModelForSequenceClassification.from_pretrained(model_id, config=cfg)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{exp_id}: {model_id}")
    print(f"  parameters : {n_params:,} ({sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable)")
    print(f"  max_length : {max_length}   text_mode: {text_mode or CONFIG['preprocess']['text_mode']}")
    _b = datasets["train"][0]
    print(f"  input ids  : {tuple(_b['input_ids'].shape)}  ->  logits (batch, {len(LABELS)})")

    out_dir = PATHS["models"] / exp_id
    out_dir.mkdir(parents=True, exist_ok=True)
    if IN_COLAB:
        _scratch = Path("/content/ckpt")
    elif IN_KAGGLE:
        _scratch = Path("/kaggle/temp/ckpt")     # scratch, not counted in kernel output
    else:
        _scratch = Path(tempfile.gettempdir()) / "ckpt"
    ckpt_dir = _scratch / exp_id
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    args = TrainingArguments(
        output_dir=str(ckpt_dir),
        learning_rate=tcfg["lr"],
        per_device_train_batch_size=tcfg["per_device_batch_size"],
        per_device_eval_batch_size=tcfg["per_device_batch_size"] * 2,
        gradient_accumulation_steps=tcfg["grad_accum_steps"],
        num_train_epochs=tcfg["epochs"],
        warmup_ratio=tcfg["warmup_ratio"],
        weight_decay=tcfg["weight_decay"],
        lr_scheduler_type=tcfg["lr_scheduler_type"],
        fp16=tcfg["fp16"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=50,
        report_to=[],
        seed=CONFIG["seed"],
        dataloader_pin_memory=torch.cuda.is_available(),
        # Speed only: batches articles of similar length together.
        group_by_length=True,
        # Halve the batch instead of crashing if it does not fit.
        auto_find_batch_size=True,
        dataloader_num_workers=2,
        max_steps=20 if CONFIG["fast_mode"] else -1,
    )
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=datasets["train"],
        eval_dataset=datasets["val"], compute_metrics=hf_metrics,
        data_collator=DataCollatorWithPadding(tok),
        class_weights=class_weights_tensor() if tcfg.get("class_weighted_loss") else None)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    train_seconds = time.time() - t0
    # train_batch_size = per-device batch x number of GPUs (Kaggle's T4 x2 runs DataParallel).
    _effective = trainer.args.train_batch_size * trainer.args.gradient_accumulation_steps
    _planned = tcfg["per_device_batch_size"] * tcfg["grad_accum_steps"]
    if _effective != _planned:
        print(f"  !! Effective batch is {_effective} (per-device "
              f"{trainer.args.per_device_train_batch_size} x {max(1, trainer.args.n_gpu)} GPU(s) "
              f"x accum {trainer.args.gradient_accumulation_steps}), not the planned {_planned} "
              f"— this IS a hyperparameter change and is recorded in the run JSON.")

    # group_by_length also reorders predict() output, so turn it off before predicting.
    trainer.args.group_by_length = False
    va_out = trainer.predict(datasets["val"])
    assert np.array_equal(va_out.label_ids, va_df["y"].values), "val predictions out of order"
    va_pred = np.argmax(va_out.predictions, axis=1)
    t1 = time.time()
    te_out = trainer.predict(datasets["test"])
    assert np.array_equal(te_out.label_ids, te_df["y"].values), "test predictions out of order"
    inference_ms = (time.time() - t1) / len(te_df) * 1000
    te_logits = te_out.predictions
    te_pred = np.argmax(te_logits, axis=1)
    proba = torch.softmax(torch.tensor(te_logits), dim=1).numpy()

    rec = log_run(
        exp_id=exp_id, name=name, family="transformer", rq=rq, description=description,
        hyperparams={"model_id": model_id, "lr": tcfg["lr"], "epochs": tcfg["epochs"],
                     "max_length": max_length,
                     "batch_size": trainer.args.per_device_train_batch_size,
                     "batch_size_requested": tcfg["per_device_batch_size"],
                     "n_gpu": trainer.args.n_gpu,
                     "effective_batch": trainer.args.train_batch_size
                                        * trainer.args.gradient_accumulation_steps,
                     "group_by_length": True,
                     "class_weighted_loss": bool(tcfg.get("class_weighted_loss")),
                     "grad_accum": tcfg["grad_accum_steps"], "warmup_ratio": tcfg["warmup_ratio"],
                     "weight_decay": tcfg["weight_decay"], "fp16": tcfg["fp16"],
                     "text_mode": text_mode or CONFIG["preprocess"]["text_mode"]},
        y_val_true=va_df["y"].values, y_val_pred=va_pred,
        y_test_true=te_df["y"].values, y_test_pred=te_pred,
        test_proba=proba, test_texts=te_df["title"].values,
        train_seconds=round(train_seconds, 1), n_params=n_params,
        inference_ms=round(inference_ms, 3), notes=notes,
    )
    trainer.save_model(str(out_dir / "best"))
    tok.save_pretrained(str(out_dir / "best"))
    print(f"  checkpoint -> {out_dir / 'best'}")
    return trainer, tok, rec


print(f"\nTransformer settings: {CONFIG['transformer']}")

# Without a GPU, stop rather than download ~6.6 GB of models that cannot be trained here.
GPU_AVAILABLE = torch.cuda.is_available()
ALLOW_CPU_TRANSFORMERS = os.environ.get("ALLOW_CPU_TRANSFORMERS", "0") == "1"

if not GPU_AVAILABLE and not CONFIG["fast_mode"] and not ALLOW_CPU_TRANSFORMERS:
    raise RuntimeError(
        "E3/E4 need a GPU.\n"
        "  No CUDA device was found, and fetching xlm-roberta-base + afro-xlmr-base plus\n"
        "  their checkpoints would need roughly 6.6 GB of disk for models that cannot be\n"
        "  trained here in reasonable time.\n\n"
        "  Run the Transformer sections on Colab (see COLAB_RUNBOOK.md), or:\n"
        "    - set CONFIG['fast_mode']=True to smoke-test the code path with a tiny stand-in\n"
        "    - set ALLOW_CPU_TRANSFORMERS=1 to override this guard deliberately"
    )
if not GPU_AVAILABLE:
    print("\n!! No GPU. Running the Transformer code path with the tiny stand-in model.")
    print("   Scores here are meaningless; the reportable runs happen on Colab.")

### 4.3 E3 — XLM-R base (the general multilingual control)

Fine-tunes XLM-R for three epochs with the settings in 4.1, keeps the checkpoint with the
best validation macro-F1, and scores it on test. It is the control: a strong multilingual
model with no Kinyarwanda in its pretraining.

In [ ]:
if not is_done("E3"):
    _tr_e3, _tok_e3, _rec_e3 = train_transformer(
        "E3", CONFIG["transformer"]["models"]["E3"], "XLM-R base (fine-tuned)", "RQ3",
        "General multilingual encoder fine-tuned for 14-way topic classification.",
        notes="RQ3 control: multilingual pretraining without African-specific adaptation.",
    )

### 4.4 E4 — AfroXLMR base (African-centric adaptation)

Identical code and settings to E3; only the pretrained weights change. AfroXLMR was adapted
to 17 African languages, Kinyarwanda among them.

In [ ]:
if not is_done("E4"):
    _tr_e4, _tok_e4, _rec_e4 = train_transformer(
        "E4", CONFIG["transformer"]["models"]["E4"], "AfroXLMR base (fine-tuned)", "RQ3",
        "XLM-R adapted to 17 African languages via MAFT, then fine-tuned on KINNEWS.",
        notes="RQ3 treatment: identical setup to E3, identical tokenizer, adapted weights.",
    )

### 4.5 What we learned from the Transformers (RQ3)

Generated from `E3.json` and `E4.json`. Tokenizer and settings are identical, so any gap
reflects AfroXLMR's exposure to Kinyarwanda.

In [ ]:
explain("E3", "E4",
        "RQ3: does African-centric adaptation beat general multilingual pretraining?",
        "AfroXLMR is XLM-R after multilingual adaptive fine-tuning on 17 African languages, "
        "including Kinyarwanda. XLM-R never saw Kinyarwanda (verified: absent from CC-100's "
        "116 languages and from its model card's 94). Both share a tokenizer with fertility "
        "2.47, so any gap comes from the adapted weights alone.")

## 5. Ablations

Both run on the better of E3/E4, chosen on validation:

- **A1 — title only:** how much of the topic signal is in the headline, which is often all
  a user pastes into the app.
- **A2 — max length 256:** what harsher truncation costs, given that 94.9% of articles
  exceed 256 tokens and 74.5% exceed 512.

In [ ]:
def best_transformer_id():
    """Pick the stronger Transformer on VALIDATION — never on test."""
    available = [i for i in ("E3", "E4") if (runs_dir() / f"{i}.json").exists()]
    if not available:
        return None
    scored = []
    for i in available:
        with open(runs_dir() / f"{i}.json") as fh:
            d = json.load(fh)
        scored.append((d["val_metrics"]["macro_f1"], i, d["hyperparameters"]["model_id"]))
    scored.sort(reverse=True)
    print(f"ablation base: {scored[0][1]} (val macro-F1 {scored[0][0]:.4f})")
    return scored[0][1], scored[0][2]


_base = best_transformer_id()
if _base:
    _bid, _bmodel = _base
    if not is_done("A1"):
        train_transformer("A1", _bmodel, f"{_bid} — title only", "ablation",
                          "Same model and settings, but only the headline is given as input.",
                          text_mode=CONFIG["ablations"]["A1_text_mode"],
                          notes="A1: measures how much topic signal lives in the title alone.")
    if not is_done("A2"):
        train_transformer("A2", _bmodel, f"{_bid} — max_length 256", "ablation",
                          "Same model and settings, but input truncated at 256 subword tokens.",
                          max_length=CONFIG["ablations"]["A2_max_length"],
                          notes="A2: measures the cost of harsher truncation.")

### 5.1 What we learned from the ablations

Each comparison is the base model against its modified version.

In [ ]:
_base_id = best_transformer_id()
if _base_id:
    explain(_base_id[0], "A1",
            "A1: how much topic signal is in the headline alone?",
            "News headlines are written to convey the subject, and Section 1 showed the "
            "model never sees past token 512 of most articles anyway. If title-only comes "
            "close to the full model, most of the usable signal is in the first line — which "
            "matters practically, because headlines are how users usually meet news.")
    print()
    explain(_base_id[0], "A2",
            "A2: what does harsher truncation cost?",
            "94.9% of articles exceed 256 subword tokens and 74.5% exceed 512, so both "
            "settings truncate most of the corpus. This measures whether the second 256 "
            "tokens carry information the first 256 did not.")
print()
_ = build_results_table()

### 5.2 Seed variance

Every comparison so far is one training run. This re-trains E2a, E2b, E3 and E4 with
seeds 43 and 44, reports test macro-F1 as mean ± std over three seeds, and checks whether
each RQ's winner wins on every seed. The fastText vectors are reused, so for E2b the seeds
vary only the BiLSTM. The runs are saved in `results/runs/seeds/`, separate from the main
results (about 45 extra GPU minutes).

In [ ]:
import gc
import shutil

_sv = CONFIG["seed_variance"]
_main_seed = CONFIG["seed"]
_trainers = {
    "E2a": lambda eid: train_bilstm(eid, "BiLSTM (random embeddings)",
                                    "Seed-variance rerun of E2a.", pretrained=None),
    "E2b": lambda eid: train_bilstm(eid, "BiLSTM (fastText embeddings)",
                                    "Seed-variance rerun of E2b.", pretrained=_matrix),
    "E3": lambda eid: train_transformer(eid, CONFIG["transformer"]["models"]["E3"],
                                        "XLM-R base (fine-tuned)", "RQ3", "Seed-variance rerun of E3."),
    "E4": lambda eid: train_transformer(eid, CONFIG["transformer"]["models"]["E4"],
                                        "AfroXLMR base (fine-tuned)", "RQ3", "Seed-variance rerun of E4."),
}
try:
    RUN_SUBDIR = "seeds"
    for _seed in _sv["extra_seeds"]:
        CONFIG["seed"] = _seed
        for _eid in _sv["experiments"]:
            _run_id = f"{_eid}_s{_seed}"
            if _eid == "E2b" and "_matrix" not in globals():
                print(f"{_run_id}: skipped (fastText matrix not in memory; run Section 3.4 first)")
                continue
            if is_done(_run_id):
                continue
            print(f"\n===== {_run_id} (seed {_seed}) =====")
            _out = _trainers[_eid](_run_id)
            del _out
            # The extra Transformer checkpoints are not used later; do not keep 1.1 GB each.
            shutil.rmtree(PATHS["models"] / _run_id, ignore_errors=True)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
finally:
    RUN_SUBDIR = ""
    CONFIG["seed"] = _main_seed
    set_seed(_main_seed)

**Seed-variance results.** Test macro-F1 per model for each seed, with mean ± std,
and whether E2b beats E2a and E4 beats E3 on every seed. Saved to
`results/seed_variance.json`.

In [ ]:
_rows = []
for _eid in _sv["experiments"]:
    _scores = {}
    if (runs_dir() / f"{_eid}.json").exists():
        _scores[_main_seed] = json.load(open(runs_dir() / f"{_eid}.json"))["test_metrics"]["macro_f1"]
    for _seed in _sv["extra_seeds"]:
        _p = runs_dir() / "seeds" / f"{_eid}_s{_seed}.json"
        if _p.exists():
            _scores[_seed] = json.load(open(_p))["test_metrics"]["macro_f1"]
    if _scores:
        _v = np.array(list(_scores.values()))
        _rows.append({"id": _eid, "seeds": len(_v),
                      **{f"seed {s}": round(x, 4) for s, x in sorted(_scores.items())},
                      "mean": round(float(_v.mean()), 4),
                      "std": round(float(_v.std(ddof=1)), 4) if len(_v) > 1 else None,
                      "min–max": f"{_v.min():.4f}–{_v.max():.4f}"})
SEED_TABLE = pd.DataFrame(_rows)
print("Test macro-F1 across seeds:\n")
print(SEED_TABLE.to_string(index=False))


def _paired(a, b):
    """For each seed where both ran, did b beat a?"""
    wins, n = 0, 0
    for s in [_main_seed] + _sv["extra_seeds"]:
        pa = runs_dir() / (f"{a}.json" if s == _main_seed else f"seeds/{a}_s{s}.json")
        pb = runs_dir() / (f"{b}.json" if s == _main_seed else f"seeds/{b}_s{s}.json")
        if pa.exists() and pb.exists():
            n += 1
            wins += json.load(open(pb))["test_metrics"]["macro_f1"] > json.load(open(pa))["test_metrics"]["macro_f1"]
    return wins, n


_seed_summary = {"table": _rows, "fasttext_fingerprint": globals().get("FASTTEXT_FINGERPRINT")}
for _a, _b, _q in (("E2a", "E2b", "RQ2"), ("E3", "E4", "RQ3")):
    _w, _n = _paired(_a, _b)
    _seed_summary[f"{_q}_{_b}_beats_{_a}"] = f"{_w}/{_n} seeds"
    if _n:
        print(f"\n{_q}: {_b} beats {_a} on {_w} of {_n} seeds"
              + ("  -> the direction of the result is stable across seeds." if _w == _n
                 else "  -> NOT stable across seeds; treat the single-seed gap with caution."))
with open(PATHS["results"] / "seed_variance.json", "w") as fh:
    json.dump(_seed_summary, fh, indent=2)
print(f"\nsaved -> {(PATHS['results'] / 'seed_variance.json').relative_to(PATHS['root'])}")

## 6. Evaluation of the selected model

The final model is the one with the highest **validation** macro-F1; test scores are never
used to choose. This section reports its per-class results, confusion matrix, significance
tests and inference speed.

In [ ]:
def load_all_runs():
    out = {}
    for path in sorted(runs_dir().glob("*.json")):
        with open(path) as fh:
            d = json.load(fh)
        out[d["id"]] = d
    return out


RUNS = load_all_runs()
print(f"experiments available: {', '.join(sorted(RUNS))}\n")

_ranked = sorted(RUNS.values(), key=lambda d: -d["val_metrics"]["macro_f1"])
print(f"{'id':6}{'model':34}{'val macro-F1':>14}{'test macro-F1':>15}")
for d in _ranked:
    print(f"{d['id']:6}{d['name'][:32]:34}{d['val_metrics']['macro_f1']:14.4f}"
          f"{d['test_metrics']['macro_f1']:15.4f}")

BEST = _ranked[0]
print(f"\nSelected on validation: {BEST['id']} — {BEST['name']}")
print(f"  test macro-F1 {BEST['test_metrics']['macro_f1']:.4f} "
      f"[95% CI {BEST['test_macro_f1_ci95'][0]:.4f}–{BEST['test_macro_f1_ci95'][1]:.4f}]")
CONFIG["best_model_id"] = BEST["id"]

### 6.1 Per-class performance

Macro-F1 is an average over 14 classes; this shows which classes actually work, next to
how much training data each had.

In [ ]:
_pc = pd.DataFrame(BEST["per_class_test"])
_pc["train_n"] = [int((tr_df.y == i).sum()) for i in range(len(LABELS))]
_pc = _pc[["class", "train_n", "support", "precision", "recall", "f1"]]
print(f"{BEST['id']} — per-class test metrics (sorted by F1):\n")
print(_pc.sort_values("f1", ascending=False).to_string(index=False))
print(f"\nmacro-F1 {BEST['test_metrics']['macro_f1']:.4f}   "
      f"weighted-F1 {BEST['test_metrics']['weighted_f1']:.4f}   "
      f"accuracy {BEST['test_metrics']['accuracy']:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(_pc.train_n, _pc.f1, s=60, color="#2980b9")
for _, r in _pc.iterrows():
    ax.annotate(r["class"], (r.train_n, r.f1), fontsize=8,
                xytext=(4, 3), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("Training articles (log scale)")
ax.set_ylabel("Test F1")
ax.set_title(f"{BEST['id']}: per-class F1 against training-set size")
save_fig(fig, "06_f1_vs_train_size")
plt.show()

### 6.2 Confusion matrices — baseline vs final model

Row-normalised, so the diagonal is per-class recall. The off-diagonal cells drive the error
analysis.

In [ ]:
_cmp_id = "B1" if "B1" in RUNS else sorted(RUNS)[0]
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
for ax, _id in zip(axes, [_cmp_id, BEST["id"]]):
    _p = load_preds(_id)
    _cm = confusion_matrix([LABELS.index(x) for x in _p["true"]],
                           [LABELS.index(x) for x in _p["predicted"]],
                           labels=ALL_LABELS, normalize="true")
    sns.heatmap(_cm, annot=True, fmt=".2f", cmap="Blues", cbar=False,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax, annot_kws={"size": 7})
    _m = RUNS[_id]["test_metrics"]["macro_f1"]
    ax.set_title(f"{_id} — {RUNS[_id]['name']}\nmacro-F1 {_m:.4f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(labelsize=8)
plt.tight_layout()
save_fig(fig, "07_confusion_matrices")
plt.show()

### 6.3 McNemar's test

Compares two models on the same test articles using only the cases where exactly one of
them is right: `(|b − c| − 1)² / (b + c)` follows χ² with 1 degree of freedom (McNemar,
1947); an exact binomial test is used when `b + c` < 25.

In [ ]:
# `mcnemar` and `load_preds` are defined in Section 3, next to `explain()`, which uses them.
_tests = []
for _a, _b in [(_cmp_id, BEST["id"]), ("E3", "E4"), ("B1", "E1"), ("E2a", "E2b")]:
    if _a in RUNS and _b in RUNS and _a != _b:
        _tests.append(mcnemar(_a, _b))
for _t in _tests:
    if "note" in _t:
        print(f"{_t['comparison']:16} {_t['note']}")
    else:
        _verdict = "SIGNIFICANT" if _t["significant_at_0.05"] else "not significant"
        print(f"{_t['comparison']:16} only-A {_t['only_A_right']:4}  only-B {_t['only_B_right']:4}"
              f"   p={_t['p_value']:.2e}   {_verdict}  ({_t['test']})")
with open(PATHS["results"] / "significance_tests.json", "w") as fh:
    json.dump(_tests, fh, indent=2)
print(f"\nsaved -> {(PATHS['results'] / 'significance_tests.json').relative_to(PATHS['root'])}")

### 6.4 Inference speed

The app runs E1 on the CPU, so time per article decides whether it feels usable. The classical
and BiLSTM models were timed on the CPU and the Transformers on the GPU, so the "CPU" in the
printout below applies to the non-Transformer rows only.

In [ ]:
_lat = {d["id"]: d["inference_ms_per_article"] for d in RUNS.values()
        if d.get("inference_ms_per_article")}
print("Measured CPU inference, milliseconds per article (batched):\n")
for _id, _ms in sorted(_lat.items(), key=lambda kv: kv[1]):
    print(f"  {_id:5} {RUNS[_id]['name'][:34]:36} {_ms:8.3f} ms")
print("\nNote: batched throughput. Single-article latency in the app is higher because there")
print("is no batching to amortise the forward pass; Section 8 measures that case directly.")

## 7. Error analysis

What the selected model gets wrong, and why, using `results/predictions/<best>.csv`.

### 7.1 Most confused class pairs

The largest off-diagonal cells, with example articles.

In [ ]:
_bp = load_preds(BEST["id"])
_bp["true_i"] = [LABELS.index(x) for x in _bp["true"]]
_bp["pred_i"] = [LABELS.index(x) for x in _bp["predicted"]]
_cm = confusion_matrix(_bp.true_i, _bp.pred_i, labels=ALL_LABELS)

_pairs = [(_cm[i, j], LABELS[i], LABELS[j], i, j)
          for i in ALL_LABELS for j in ALL_LABELS if i != j and _cm[i, j] > 0]
_pairs.sort(reverse=True)
print("Most confused pairs (true -> predicted):\n")
for _n, _t, _p, _i, _j in _pairs[:8]:
    _share = _n / max(_cm[_i].sum(), 1) * 100
    print(f"  {_t:14} -> {_p:14} {_n:4} articles  ({_share:.1f}% of all true {_t})")

print("\n" + "=" * 100)
for _n, _t, _p, _i, _j in _pairs[:3]:
    print(f"\nTRUE {_t}  ->  PREDICTED {_p}   ({_n} articles)")
    _ex = _bp[(_bp.true == _t) & (_bp.predicted == _p)]
    if "top1_prob" in _ex.columns:
        _ex = _ex.sort_values("top1_prob", ascending=False)
    for _, _r in _ex.head(3).iterrows():
        _conf = f"{_r['top1_prob']:.3f}" if "top1_prob" in _r else "n/a"
        print(f"   [conf {_conf}] {str(_r.get('snippet', ''))[:95]}")

### 7.2 Performance slices

By article length (macro-F1 over the classes present in each group) and by the training
size of the true class (accuracy, because each size group contains only a few classes).

In [ ]:
_bp["n_words"] = te_df["n_words"].values
_buckets = CONFIG["eval"]["length_buckets"]
_labels_b = [f"{_buckets[i]}–{_buckets[i+1] if _buckets[i+1] < 10**8 else '∞'}"
             for i in range(len(_buckets) - 1)]
_bp["len_bucket"] = pd.cut(_bp.n_words, bins=_buckets, labels=_labels_b, right=False)

_train_counts = tr_df.y.value_counts()
def _size_band(cls):
    n = int(_train_counts.get(LABELS.index(cls), 0))
    return "small (<100)" if n < 100 else ("medium (100–499)" if n < 500 else "large (500+)")
_bp["size_band"] = _bp["true"].map(_size_band)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col, title, metric in ((axes[0], "len_bucket", "by article length (words)", "macro_f1"),
                               (axes[1], "size_band", "by training-set size of the true class", "accuracy")):
    rows = []
    for g, sub in _bp.groupby(col, observed=True):
        if len(sub) < 5:
            continue
        # macro-F1 over the classes present in the group only.
        present = sorted(sub.true_i.unique())
        rows.append({"group": str(g), "macro_f1": f1_score(sub.true_i, sub.pred_i,
                     average="macro", labels=present, zero_division=0),
                     "accuracy": (sub.true_i == sub.pred_i).mean(), "n": len(sub)})
    _d = pd.DataFrame(rows)
    ax.bar(_d.group, _d[metric], color="#16a085")
    for i, r in _d.iterrows():
        ax.text(i, r[metric] + 0.01, f"n={r.n}", ha="center", fontsize=8)
    _metric_name = "macro-F1" if metric == "macro_f1" else "accuracy"
    ax.set_title(f"{BEST['id']} {_metric_name} {title}")
    ax.set_ylabel(_metric_name)
    ax.tick_params(axis="x", rotation=20)
    print(f"\n{title}:\n{_d.to_string(index=False)}")
plt.tight_layout()
save_fig(fig, "08_performance_slices")
plt.show()

### 7.3 Confident mistakes

Errors made with high confidence usually point to label noise or an article that belongs
to two topics. Section 1.3 found 156 duplicate groups with conflicting labels, so label
noise is known to exist.

In [ ]:
if "top1_prob" in _bp.columns:
    _conf_err = _bp[~_bp.correct].sort_values("top1_prob", ascending=False).head(
        CONFIG["eval"]["n_confident_mistakes"])
    print(f"The {len(_conf_err)} highest-confidence errors:\n")
    for _, _r in _conf_err.iterrows():
        print(f"  conf {_r.top1_prob:.3f}  true={_r.true:14} pred={_r.predicted:14}")
        print(f"            {str(_r.snippet)[:92]}")
    _conf_err.to_csv(PATHS["audit"] / "confident_mistakes.csv", index=False)
    print(f"\nsaved -> {(PATHS['audit'] / 'confident_mistakes.csv').relative_to(PATHS['root'])}")
    print("\nMarked CANDIDATE label noise — confirming any of these requires reading")
    print("Kinyarwanda, which is the student's judgement to make, not the code's.")

### 7.4 Where the model families disagree

Articles the baseline gets right and the final model gets wrong, and the reverse.

In [ ]:
if _cmp_id != BEST["id"]:
    best_id = BEST["id"]
    ok_a, ok_b = f"{_cmp_id}_ok", f"{best_id}_ok"
    pred_a, pred_b = f"{_cmp_id}_pred", f"{best_id}_pred"

    _op = load_preds(_cmp_id)
    both = pd.DataFrame({
        "true": _bp["true"].values,
        "snippet": _bp["snippet"].values if "snippet" in _bp else "",
        pred_a: _op["predicted"].values,
        ok_a: _op["correct"].values.astype(bool),
        pred_b: _bp["predicted"].values,
        ok_b: _bp["correct"].values.astype(bool),
    })

    a_only = both[both[ok_a] & ~both[ok_b]]
    b_only = both[~both[ok_a] & both[ok_b]]
    print(f"{_cmp_id} right, {best_id} wrong : {len(a_only):5}")
    print(f"{best_id} right, {_cmp_id} wrong : {len(b_only):5}")
    print(f"both right                 : {int((both[ok_a] & both[ok_b]).sum()):5}")
    print(f"both wrong                 : {int((~both[ok_a] & ~both[ok_b]).sum()):5}")

    for title, subset in ((f"{_cmp_id} right, {best_id} wrong", a_only),
                          (f"{best_id} right, {_cmp_id} wrong", b_only)):
        print(f"\n--- {title} (3 examples) ---")
        for _, r in subset.head(3).iterrows():
            print(f"  true={r['true']:14} {_cmp_id}={r[pred_a]:14} {best_id}={r[pred_b]:14}")
            print(f"    {str(r['snippet'])[:90]}")

### 7.5 Candidate error causes

Simple heuristics tag a sample of errors with possible causes (topic overlap, short text,
truncation, code-switching, proper names, possible label noise). They are candidates only;
7.5b is where a Kinyarwanda reader confirms them.

In [ ]:
OVERLAP_PAIRS = {frozenset(p) for p in [("politics", "economy"), ("politics", "history"),
                                        ("economy", "environment"), ("culture", "history"),
                                        ("entertainment", "fashion"), ("culture", "religion"),
                                        ("health", "environment"), ("education", "politics")]}
LATIN_RE = re.compile(r"\b(the|and|for|with|from|this|that|will|have|said|dans|pour|avec|les|des)\b",
                      flags=re.I)


def classify_error(row):
    causes = []
    snip = str(row.get("snippet", ""))
    if frozenset((row["true"], row["predicted"])) in OVERLAP_PAIRS:
        causes.append("topic overlap")
    if row.get("n_words", 999) < 60:
        causes.append("very short text")
    if row.get("n_words", 0) > 512:
        causes.append("truncation (beyond max_length)")
    if len(LATIN_RE.findall(snip)) >= 2:
        causes.append("code-switching (English/French)")
    if sum(c.isupper() for c in snip) > len(snip) * 0.12:
        causes.append("rare vocabulary / proper names")
    if row.get("top1_prob", 0) > 0.85:
        causes.append("possible label noise (high confidence)")
    return causes or ["unclassified"]


_errs = _bp[~_bp.correct].copy()
_sample = _errs.sample(min(CONFIG["eval"]["n_taxonomy_errors"], len(_errs)),
                       random_state=CONFIG["seed"]) if len(_errs) else _errs
if len(_sample):
    _sample["candidate_causes"] = _sample.apply(lambda r: "; ".join(classify_error(r)), axis=1)
    from collections import Counter
    _counts = Counter(c for r in _sample.itertuples()
                      for c in r.candidate_causes.split("; "))
    print(f"Candidate causes across {len(_sample)} sampled errors "
          f"(of {len(_errs)} total). Categories overlap.\n")
    for _c, _n in _counts.most_common():
        print(f"  {_n:4}  {_c}")
    _sample[["true", "predicted", "candidate_causes", "snippet"]].to_csv(
        PATHS["audit"] / "error_taxonomy.csv", index=False)
    print(f"\nsaved -> {(PATHS['audit'] / 'error_taxonomy.csv').relative_to(PATHS['root'])}")
    print("ALL ROWS ARE CANDIDATES — student to confirm.")

### 7.5b Manual review sheet

Collects the label spot checks (1.3), confident mistakes (7.3) and sampled errors (7.5)
into `data/audit/manual_review_TEMPLATE.csv`. Copy it to `data/audit/manual_review.csv`,
fill in `verdict` (`label_correct` / `label_wrong` / `ambiguous`), `correct_label` and, for
errors, `cause`, then run `python scripts/summarise_manual_review.py`.

In [ ]:
_review = []
for _r in spot_check.itertuples():
    _review.append({"source": "label_spot_check (train)", "row": _r.row, "given_label": _r.label,
                    "model_predicted": "", "title_or_snippet": f"{_r.title} | {_r.snippet}"})
if "_conf_err" in globals():
    for _r in _conf_err.itertuples():
        _review.append({"source": "confident_mistake (test)", "row": _r.id,
                        "given_label": _r.true, "model_predicted": _r.predicted,
                        "title_or_snippet": str(_r.snippet)})
if len(_sample):
    for _r in _sample.itertuples():
        _review.append({"source": "sampled_error (test)", "row": _r.id,
                        "given_label": _r.true, "model_predicted": _r.predicted,
                        "title_or_snippet": str(_r.snippet),
                        "heuristic_causes": _r.candidate_causes})
REVIEW = pd.DataFrame(_review)
for _col in ("verdict", "correct_label", "cause", "note"):
    REVIEW[_col] = ""
REVIEW.to_csv(PATHS["audit"] / "manual_review_TEMPLATE.csv", index=False)
print(f"{len(REVIEW)} rows for manual review "
      f"({REVIEW.source.value_counts().to_dict()})")
print(f"saved -> {(PATHS['audit'] / 'manual_review_TEMPLATE.csv').relative_to(PATHS['root'])}")

### 7.6 Out-of-domain probes

A softmax classifier always names a topic, even for English, French, Kirundi or random
text. These probes show it, which is why the app warns on low confidence.

In [ ]:
PROBES = [
    ("Kinyarwanda, social-media register", "Yooo ndabona iyi ngoma ni nziza cyane 😂 murakoze"),
    ("English news sentence", "The central bank raised interest rates by half a point today."),
    ("French news sentence", "Le president a annonce une nouvelle reforme economique hier."),
    ("Kirundi (closely related language)", "Abanyagihugu barashima ivyo bakoze mu gihugu cabo."),
    ("Random characters", "xkcd qzmp vvfl aaaa zzzz qqqq"),
    ("Empty-ish input", "..."),
]


def predict_texts(texts):
    """Score raw strings with the selected model, whatever family it belongs to."""
    if BEST["family"] == "classical":
        vec, clf = (vec_e1, clf_e1) if BEST["id"] == "E1" else (vec_b1, clf_b1)
        X = vec.transform([preprocess_classical(t) for t in texts])
        return clf.predict_proba(X)
    if BEST["family"] == "recurrent":
        model = model_e2b if BEST["id"] == "E2b" else model_e2a
        stoi = stoi_e2b if BEST["id"] == "E2b" else stoi_e2a
        ds = NewsDataset([preprocess_classical(t) for t in texts], [0] * len(texts),
                         stoi, CONFIG["bilstm"]["max_tokens"])
        loader = DataLoader(ds, batch_size=8)
        model.eval()
        with torch.no_grad():
            return torch.cat([torch.softmax(model(i.to(DEVICE), m.to(DEVICE)), 1).cpu()
                              for i, m, _ in loader]).numpy()
    trainer = _tr_e4 if BEST["id"] in ("E4", "A1", "A2") else _tr_e3
    tok = _tok_e4 if BEST["id"] in ("E4", "A1", "A2") else _tok_e3
    enc = dict(tok([preprocess_transformer(t) for t in texts], truncation=True,
                   max_length=CONFIG["transformer"]["max_length"], padding=True))
    logits = trainer.predict(HFDataset(enc, [0] * len(texts))).predictions
    return torch.softmax(torch.tensor(logits), dim=1).numpy()


try:
    _probe_proba = predict_texts([t for _, t in PROBES])
    print(f"Out-of-domain probes against {BEST['id']}:\n")
    _rows = []
    for (_desc, _text), _pr in zip(PROBES, _probe_proba):
        _top = np.argsort(-_pr)[:3]
        _rows.append({"probe": _desc, "text": _text[:48],
                      "top1": LABELS[_top[0]], "conf": round(float(_pr[_top[0]]), 3),
                      "top2": LABELS[_top[1]], "top3": LABELS[_top[2]]})
        print(f"  {_desc:36} -> {LABELS[_top[0]]:14} ({_pr[_top[0]]:.3f})")
    pd.DataFrame(_rows).to_csv(PATHS["audit"] / "ood_probes.csv", index=False)
    print(f"\nsaved -> {(PATHS['audit'] / 'ood_probes.csv').relative_to(PATHS['root'])}")
    _maxc = max(r["conf"] for r in _rows)
    print(f"\nHighest confidence on any non-Kinyarwanda-news input: {_maxc:.3f}")
    print("The model names a topic for every one of these. That is the behaviour the app's")
    print("low-confidence notice exists to communicate.")
except Exception as _e:
    print(f"probe skipped ({type(_e).__name__}: {_e})")

### 7.7 Low-confidence threshold for the app

The probability that best separates correct from incorrect predictions (Youden's J),
chosen on **validation** so the test results stay untouched.

In [ ]:
_thr_path = PATHS["results"] / "confidence_threshold.json"
_vp = None
if BEST["family"] == "classical" and BEST["id"] in ("B1", "E1"):
    _vec, _clf = (vec_e1, clf_e1) if BEST["id"] == "E1" else (vec_b1, clf_b1)
    _vp = _clf.predict_proba(_vec.transform(va_df["text_classical"]))
if _vp is not None:
    _vconf = _vp.max(axis=1)
    _vcorrect = (_vp.argmax(axis=1) == va_df["y"].values)
    _best_t, _best_j = 0.5, -1
    for _t in np.arange(0.20, 0.95, 0.01):
        _tpr = (_vconf[_vcorrect] >= _t).mean()
        _fpr = (_vconf[~_vcorrect] >= _t).mean()
        if _tpr - _fpr > _best_j:
            _best_t, _best_j = float(_t), _tpr - _fpr
    _flagged = float((_vconf < _best_t).mean())
    _acc_above = float(_vcorrect[_vconf >= _best_t].mean())
    _acc_below = float(_vcorrect[_vconf < _best_t].mean()) if (_vconf < _best_t).any() else float("nan")
    print(f"threshold {_best_t:.2f} (Youden J = {_best_j:.3f}) chosen on VALIDATION")
    print(f"  flags {_flagged*100:.1f}% of validation predictions as low-confidence")
    print(f"  accuracy above threshold: {_acc_above:.3f}")
    print(f"  accuracy below threshold: {_acc_below:.3f}")
    with open(_thr_path, "w") as fh:
        json.dump({"model": BEST["id"], "threshold": _best_t, "youden_j": _best_j,
                   "val_flagged_fraction": _flagged, "val_acc_above": _acc_above,
                   "val_acc_below": _acc_below}, fh, indent=2)
    print(f"saved -> {_thr_path.relative_to(PATHS['root'])}")
else:
    print(f"threshold selection needs in-memory model objects for {BEST['id']};")
    print("run Sections 3/4 in the same session, or the app falls back to 0.50.")

## 8. Export for the web app

Saves the model, its threshold and example inputs for the app. How a request flows
through the app is described in `app/README.md`.

In [ ]:
HF_USERNAME = os.environ.get("HF_USERNAME", "[[TODO: your Hugging Face username]]")
HF_MODEL_ID = f"{HF_USERNAME}/kinyarwanda-news-classifier"
print(f"target Hub repo: {HF_MODEL_ID}")
print("Set HF_USERNAME and HF_TOKEN in the environment before running the push cell.")

### 8.1 Saving the final model

For a sparse winner (B1/E1), saves the vectoriser and classifier as one scikit-learn
`Pipeline` in `app/model/`, with the exact preprocessing settings in `preprocess.json`,
and checks that the reloaded file reproduces the in-memory predictions. A Transformer
winner would be pushed to the Hugging Face Hub instead (8.2).

In [ ]:
MODEL_DIR = PATHS["root"] / "app" / "model"
if BEST["family"] == "classical" and BEST["id"] in ("B1", "E1"):
    import joblib
    import sklearn
    from sklearn.pipeline import Pipeline

    _vec, _clf = (vec_e1, clf_e1) if BEST["id"] == "E1" else (vec_b1, clf_b1)
    # stop_words_ is only for introspection and would bloat the saved file.
    _vec.stop_words_ = None
    _pipe = Pipeline([("tfidf", _vec), ("clf", _clf)])
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    _model_path = MODEL_DIR / f"{BEST['id'].lower()}_pipeline.joblib"
    joblib.dump(_pipe, _model_path, compress=3)

    # Round trip: the reloaded file must reproduce the in-memory predictions exactly.
    _check = va_df["text_classical"]
    _reloaded = joblib.load(_model_path)
    assert (_reloaded.predict(_check) == _clf.predict(_vec.transform(_check))).all(), \
        "saved pipeline disagrees with the in-memory model"

    (MODEL_DIR / "preprocess.json").write_text(json.dumps({
        "model_file": _model_path.name,
        "labels": LABELS,
        "text": "title + ' . ' + content (the app receives a single text box)",
        "unicode_normalisation": "NFC", "lowercase": True,
        "url_re": URL_RE.pattern, "email_re": EMAIL_RE.pattern,
        "num_re": NUM_RE.pattern, "keep_re": KEEP_RE.pattern,
        "url_token": CONFIG["preprocess"]["url_token"],
        "email_token": CONFIG["preprocess"]["email_token"],
        "num_token": CONFIG["preprocess"]["num_token"],
        "scikit_learn_version": sklearn.__version__,
    }, indent=2, ensure_ascii=False))
    _size_mb = _model_path.stat().st_size / 1e6
    print(f"saved -> {_model_path.relative_to(PATHS['root'])}  ({_size_mb:.1f} MB)")
    print(f"saved -> {(MODEL_DIR / 'preprocess.json').relative_to(PATHS['root'])}")
    print(f"round-trip check passed on {len(_check):,} validation articles")
else:
    print(f"Best model is {BEST['id']} ({BEST['family']}); it is exported by the Hub push in 8.2.")

### 8.2 Model card and optional Hub push

Writes a model card (intended use, data, metrics, limitations). The Hub push runs only for
a Transformer winner, and only when `HF_USERNAME` and `HF_TOKEN` are set.

In [ ]:
def _card_model_line(run: dict) -> str:
    if run["family"] == "transformer":
        return f"Fine-tuned from `{run['hyperparameters'].get('model_id', 'n/a')}`"
    return f"{run['name']}, trained"


def _card_input_limitation(run: dict) -> str:
    if run["family"] == "transformer":
        return ("**Truncation.** 74.5% of articles exceed 512 subword tokens, so the model never "
                "sees the tail of most documents.")
    return ("**Bag of character n-grams.** The model ignores word order, so it cannot use "
            "sentence structure or negation; it relies on which stems and affixes appear.")


def build_model_card(run: dict, threshold: float | None) -> str:
    per_class = "\n".join(
        f"| {c['class']} | {c['precision']:.3f} | {c['recall']:.3f} | {c['f1']:.3f} | {c['support']} |"
        for c in run["per_class_test"])
    lo, hi = run["test_macro_f1_ci95"]
    thr = f"{threshold:.2f}" if threshold else "0.50 (fallback)"
    return f"""---
language: rw
license: mit
tags: [text-classification, kinyarwanda, news, low-resource, african-nlp]
datasets: [kinnews_kirnews]
metrics: [f1, accuracy]
pipeline_tag: text-classification
---

# Kinyarwanda News Topic Classifier

Classifies a Kinyarwanda news article into one of 14 topics. {_card_model_line(run)}
on the KINNEWS corpus (Niyongabo et al., 2020).

## Intended use

Tagging, routing and organising Kinyarwanda news articles — for newsroom and archive staff,
aggregators, and researchers who need a documented Kinyarwanda baseline.

**Out of scope:** any consequential decision about a person; moderation; detecting whether
text is Kinyarwanda at all; languages other than Kinyarwanda; non-news text.

## Results ({run['id']} — {run['name']})

| Metric | Test |
|---|---|
| macro-F1 | **{run['test_metrics']['macro_f1']:.4f}** |
| 95% CI (1,000-resample bootstrap) | {lo:.4f} – {hi:.4f} |
| accuracy | {run['test_metrics']['accuracy']:.4f} |
| weighted F1 | {run['test_metrics']['weighted_f1']:.4f} |

Evaluated once on the official KINNEWS test split ({run['n_test']:,} articles). All tuning
used a held-out validation split.

### Per-class

| Class | Precision | Recall | F1 | Support |
|---|---|---|---|---|
{per_class}

## Training data

KINNEWS raw (MIT licence), 17,014 official training articles. **55% were exact duplicates**
and were removed before splitting — leaving {run['n_train']:,} training and {run['n_val']:,}
validation articles. Deduplication happens before the split so copies of one article cannot
straddle it.

## Limitations

1. **Train and test distributions differ sharply.** In the official split, environment is
   ~19x more common in test than train and fashion ~11x. Per-class scores on rare classes
   are far weaker than the headline number suggests — read the per-class table.
2. **Label noise.** 156 groups of byte-identical articles carry conflicting labels
   (1,446 rows), an irreducible ceiling on achievable accuracy.
3. **Confidently wrong out of domain.** Given English, French, Kirundi or random characters,
   the model still returns a Kinyarwanda news topic, often with high confidence. It cannot
   detect that an input is outside its domain. The demo flags predictions below {thr}.
4. {_card_input_limitation(run)}
5. **Single seed, one corpus, one time period.** No multi-seed variance estimate; news
   vocabulary drifts, so performance on recent articles may be lower.

## Citation

```bibtex
@inproceedings{{niyongabo2020kinnews,
  title={{KINNEWS and KIRNEWS: Benchmarking Cross-Lingual Text Classification for Kinyarwanda and Kirundi}},
  author={{Niyongabo, Rubungo Andre and Hong, Qu and Kreutzer, Julia and Huang, Li}},
  booktitle={{Proceedings of the 28th International Conference on Computational Linguistics}},
  year={{2020}}
}}
```
"""


_threshold = None
if (PATHS["results"] / "confidence_threshold.json").exists():
    with open(PATHS["results"] / "confidence_threshold.json") as fh:
        _threshold = json.load(fh).get("threshold")

CARD = build_model_card(BEST, _threshold)
(PATHS["results"] / "model_card.md").write_text(CARD)
print(f"model card written -> {(PATHS['results'] / 'model_card.md').relative_to(PATHS['root'])}")
print(f"  ({len(CARD.splitlines())} lines; pushed to the Hub as README.md alongside the model)")

**Hub push.** Skipped unless the winner is a Transformer and `HF_USERNAME` and
`HF_TOKEN` are set; for E1 the app loads the pipeline saved in 8.1 instead.

In [ ]:
PUSH = os.environ.get("HF_TOKEN") and "[[TODO" not in HF_MODEL_ID
if not PUSH:
    print("Skipping Hub push: set HF_USERNAME and HF_TOKEN to enable it.")
    print("This cell is the only one in the notebook that needs credentials.")
elif BEST["family"] != "transformer":
    print(f"Best model is {BEST['id']} ({BEST['family']}), which is not a Hub-native")
    print("checkpoint. Push is skipped; see app/README.md for how the app loads it instead.")
else:
    from huggingface_hub import HfApi

    _ckpt = PATHS["models"] / BEST["id"] / "best"
    (_ckpt / "README.md").write_text(CARD)
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(HF_MODEL_ID, exist_ok=True, repo_type="model")
    api.upload_folder(folder_path=str(_ckpt), repo_id=HF_MODEL_ID, repo_type="model")
    print(f"pushed -> https://huggingface.co/{HF_MODEL_ID}")

### 8.3 Single-article latency

The app classifies one article at a time, so this is the wait a user actually sees (6.4
measured batched throughput).

In [ ]:
_probe_text = te_df["text_transformer"].iloc[0] if len(te_df) else "Umukino wa Rayon Sports."
try:
    _ = predict_texts([_probe_text])            # warm up
    _t0 = time.time()
    _reps = 5
    for _ in range(_reps):
        predict_texts([_probe_text])
    _single_ms = (time.time() - _t0) / _reps * 1000
    print(f"single-article latency on {DEVICE}: {_single_ms:.1f} ms  "
          f"({_reps} runs, {BEST['id']})")
    with open(PATHS["results"] / "latency.json", "w") as fh:
        json.dump({"model": BEST["id"], "device": str(DEVICE),
                   "single_article_ms": round(_single_ms, 2),
                   "batched_ms_per_article": BEST.get("inference_ms_per_article")}, fh, indent=2)
    print(f"saved -> {(PATHS['results'] / 'latency.json').relative_to(PATHS['root'])}")
except Exception as _e:
    print(f"latency probe skipped ({type(_e).__name__}: {_e})")

### 8.4 App data files

Generates `app/model_meta.json` (metrics and threshold) and `app/examples.json` (real test
articles, including a labelled known failure), so the app never carries hand-typed
numbers.

In [ ]:
APP_DIR = PATHS["root"] / "app"
APP_DIR.mkdir(exist_ok=True)

_pc_rows = "\n".join(
    f"| {c['class']} | {c['f1']:.3f} | {c['support']} |" for c in BEST["per_class_test"])
_lo, _hi = BEST["test_macro_f1_ci95"]
_about = f"""### Results ({BEST['id']} — {BEST['name']})

| Metric | Test |
|---|---|
| macro-F1 | **{BEST['test_metrics']['macro_f1']:.4f}** |
| 95% CI | {_lo:.4f} – {_hi:.4f} |
| accuracy | {BEST['test_metrics']['accuracy']:.4f} |

Evaluated once on the official KINNEWS test split ({BEST['n_test']:,} articles).

| Class | Test F1 | Support |
|---|---|---|
{_pc_rows}
"""

_meta = {
    "model_id": BEST["id"],
    "model_name": BEST["name"],
    "threshold": _threshold if _threshold else 0.50,
    "threshold_source": "validation split (notebook 7.7)" if _threshold else "fallback default",
    "test_macro_f1": BEST["test_metrics"]["macro_f1"],
    "test_accuracy": BEST["test_metrics"]["accuracy"],
    "test_macro_f1_ci95": BEST["test_macro_f1_ci95"],
    "fast_mode": BEST["fast_mode"],
    "about_table": _about,
}
(APP_DIR / "model_meta.json").write_text(json.dumps(_meta, indent=2))
print(f"wrote app/model_meta.json  (threshold {_meta['threshold']:.2f}, "
      f"source: {_meta['threshold_source']})")

# Three correct predictions, one confident failure, one out-of-domain probe.
_examples = []
_correct = _bp[_bp.correct]
if "top1_prob" in _correct.columns:
    _correct = _correct.sort_values("top1_prob", ascending=False)
for _, _r in _correct.head(3).iterrows():
    _examples.append({"label": f"Correct: {_r['true']}", "text": str(_r["snippet"]),
                      "note": f"model predicts {_r['predicted']}"})

_fails = _bp[~_bp.correct]
if "top1_prob" in _fails.columns and len(_fails):
    _f = _fails.sort_values("top1_prob", ascending=False).iloc[0]
    _examples.append({
        "label": f"KNOWN FAILURE: true {_f['true']}, predicted {_f['predicted']}",
        "text": str(_f["snippet"]),
        "note": f"The model is confidently wrong here ({_f['top1_prob']:.0%}). "
                f"{_f['true']} and {_f['predicted']} overlap in this corpus."})

_examples.append({"label": "Out of domain: English", "note": "Still returns a topic.",
                  "text": "The central bank raised interest rates by half a point today."})

(APP_DIR / "examples.json").write_text(json.dumps(_examples, indent=2, ensure_ascii=False))
print(f"wrote app/examples.json    ({len(_examples)} examples, "
      f"{sum(1 for e in _examples if 'FAILURE' in e['label'])} known failure)")
for _e in _examples:
    print(f"  {_e['label'][:52]:54} {_e['text'][:44]}...")

if BEST["fast_mode"]:
    print("\n!! These were generated in FAST MODE. Regenerate from a full run before deploying.")

## 9. Results summary

One answer per research question, generated from `results/runs/`. Discussion, limitations,
future work and references are in the report (`report/`).

In [ ]:
def conclude():
    runs = {p.stem: json.load(open(p)) for p in runs_dir().glob("*.json")}
    if not runs:
        display(Markdown("No results yet — run the experiments first."))
        return

    def f1(i):
        return runs[i]["test_metrics"]["macro_f1"] if i in runs else None

    def line(rq, question, ids, reading):
        have = [i for i in ids if i in runs]
        if not have:
            return f"**{rq}** — {question}\n\n  *Not yet run: {', '.join(ids)}.*\n"
        scores = "; ".join(f"{i} = {f1(i):.4f}" for i in have)
        return f"**{rq}** — {question}\n\n  {scores}. {reading(have)}\n"

    parts = ["### Answers\n"]
    parts.append(line("RQ1", "Do character n-grams beat word n-grams?", ["B1", "E1"],
        lambda h: (f"Character n-grams gain {f1('E1') - f1('B1'):+.4f} macro-F1 over word "
                   f"n-grams with {runs['E1']['n_parameters'] / runs['B1']['n_parameters']:.2f}x "
                   f"the parameters." if {"B1", "E1"} <= set(h) else "Partial results.")))
    parts.append(line("RQ2", "Do in-domain fastText embeddings help a BiLSTM?", ["E2a", "E2b"],
        lambda h: (f"fastText initialisation changes macro-F1 by {f1('E2b') - f1('E2a'):+.4f}."
                   if {"E2a", "E2b"} <= set(h) else "Partial results.")))
    parts.append(line("RQ3", "Does African-centric pretraining beat general multilingual?", ["E3", "E4"],
        lambda h: (f"AfroXLMR changes macro-F1 by {f1('E4') - f1('E3'):+.4f} against XLM-R, "
                   f"using an identical tokenizer — so the difference is attributable to the "
                   f"adapted weights."
                   if {"E3", "E4"} <= set(h) else "Partial results.")))

    best = max(runs.values(), key=lambda d: d["val_metrics"]["macro_f1"])
    parts.append(f"**RQ4** — What does the best model still get wrong?\n\n"
                 f"  {best['id']} ({best['name']}) was selected on validation and reaches "
                 f"{best['test_metrics']['macro_f1']:.4f} test macro-F1 "
                 f"[{best['test_macro_f1_ci95'][0]:.4f}–{best['test_macro_f1_ci95'][1]:.4f}]. "
                 f"Section 7 breaks its errors down; the recurring themes are topic overlap "
                 f"between neighbouring classes, truncation, and irreducible label noise.\n")

    fast = any(d["fast_mode"] for d in runs.values())
    parts.append("\n---\n" + ("> **FAST MODE results present — not reportable.**"
                 if fast else "> All results above are full runs."))
    display(Markdown("\n".join(parts)))


conclude()
print()
_final = build_results_table(write=False)
if len(_final):
    print(_final[["id", "model", "RQ", "val_macro_f1", "test_macro_f1",
                  "test_ci95_low", "test_ci95_high"]].to_string(index=False))